In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Dynamic flood inundation report generator with clean two-column village tables.

Key features
------------
- Districts are detected automatically from the flood layer.
- Revenue circles and inundated villages are detected dynamically.
- RC/LULC and village statistics are calculated from GIS layers.
- District inundation map is generated automatically.
- Satellite, inundation and LULC maps are placed in a stable, non-floating layout.
- Long village tables are split into two side-by-side tables per page.
- Revenue Circle cells are vertically merged and rotated.
- Numeric columns are right aligned and displayed with two decimals.
- Each district starts on a new page.
- Word table headers repeat on continuation pages.
"""

import os
import re
import warnings
from pathlib import Path
from datetime import datetime
from io import BytesIO
import getpass
import tempfile
import subprocess



import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
from matplotlib.patches import Patch
from PIL import Image, ImageOps

from docx import Document
from docx.shared import Inches, Cm, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.section import WD_ORIENT, WD_SECTION
from docx.oxml import OxmlElement, parse_xml
from docx.oxml.ns import qn, nsdecls
from lxml import etree

import glob

import requests
from shapely.geometry import shape
from shapely.ops import unary_union
from pyproj import Transformer
from skimage.transform import ProjectiveTransform, warp
from skimage.draw import polygon

try:
    import contextily as ctx
    HAVE_CONTEXTILY = True
except Exception:
    ctx = None
    HAVE_CONTEXTILY = False

# Optional Sentinel-1 COG display support.
# Footprint generation still works if these are unavailable.
try:
    import rasterio
    from rasterio.vrt import WarpedVRT
    from rasterio.enums import Resampling
    import boto3
    from rasterio.session import AWSSession
    HAVE_SENTINEL_COG_SUPPORT = True
except Exception:
    HAVE_SENTINEL_COG_SUPPORT = False

import zipfile
# ============================================================
# 1. USER SETTINGS
# ============================================================
# ------------------------------------------------------------
# WINDOWS PATH SETTINGS
# ------------------------------------------------------------
# Edit only this block if your local folder structure is different.
#
# Recommended structure:
# D:\2026\FLOOD\DATA\
#     README.txt
#     as_*.zip
#     Shapefile_Assam\
#     Assam_LULC\
#
WINDOWS_DATA_ROOT = r"Z:\Flood Inundation\DATA"
WINDOWS_REPORT_ROOT = r"Z:\Flood Inundation\01092026"

DIST_SHP = os.path.join(
    WINDOWS_DATA_ROOT,
    "Shapefile_Assam",
    "Assam_dist_new.shp"
)

RC_SHP = os.path.join(
    WINDOWS_DATA_ROOT,
    "Shapefile_Assam",
    "Assam_RC_updated_1.shp"
)

VILL_SHP = os.path.join(
    WINDOWS_DATA_ROOT,
    "Shapefile_Assam",
    "Assam_village_updated.shp"
)

LULC_SHP = os.path.join(
    WINDOWS_DATA_ROOT,
    "Shapefile_Assam",
    "Assam_villageoLULC.shp"
)

LULC_MAP_DIR = os.path.join(
    WINDOWS_DATA_ROOT,
    "Assam_LULC"
)

FLOOD_DIR = WINDOWS_DATA_ROOT

README_PATTERN = os.path.join(
    WINDOWS_DATA_ROOT,
    "README.txt"
)


# ------------------------------------------------------------
# TITLE PAGE + FINAL PDF SETTINGS
# ------------------------------------------------------------
# Title_new.docx contains the two existing title pages.
# The original template is never overwritten.
TITLE_TEMPLATE = os.path.join(
    WINDOWS_DATA_ROOT,
    "Title_new.docx"
)

# Entered manually once at runtime. Leave as None to prompt.
REPORT_SERIAL_INPUT = None
# Sentinel-1 / CDSE raster display.
# The program first checks environment variables. If they are not present,
# it asks once for the credentials during the run.
CDSE_S3_ACCESS_KEY = None
CDSE_S3_SECRET_KEY = None

# True = draw OpenStreetMap below SAR.
USE_OSM_BASEMAP = True

# True = read actual Sentinel-1 VV/VH COG and overlay it.
USE_SENTINEL_SAR_RASTER = True

# Keep every returned Sentinel scene that intersects Assam.
AUTO_USE_ALL_RETURNED_SCENES = True

# ============================================================
# FIND FLOOD ZIP
# ============================================================
# ============================================================
# READ DATE FROM README FIRST
# ============================================================

def get_date_from_readme(readme_file):

    if not os.path.exists(readme_file):
        raise FileNotFoundError(
            f"README file not found:\n{readme_file}"
        )

    with open(
        readme_file,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:
        content = f.read()

    pattern = (
        r"Near\s+Real\s+Time\s+Inundation\s+Mapping\s*:\s*"
        r"(\d{1,2}-[A-Za-z]{3}-\d{4})"
    )

    matches = list(
        re.finditer(
            pattern,
            content,
            flags=re.IGNORECASE
        )
    )

    if not matches:
        raise RuntimeError(
            "Could not extract acquisition date from README."
        )

    # IMPORTANT:
    # Use the LAST NRT entry in README,
    # not simply the first occurrence.
    readme_date = matches[-1].group(1).strip()

    dt = datetime.strptime(
        readme_date,
        "%d-%b-%Y"
    )

    date_for_processing = dt.strftime(
        "%d-%m-%Y"
    )

    print("\n" + "=" * 70)
    print("DATE AUTOMATICALLY READ FROM README")
    print("=" * 70)
    print("README date     :", readme_date)
    print("Processing date :", date_for_processing)

    return (
        date_for_processing,
        dt
    )


DATE, EVENT_DATE = get_date_from_readme(
    README_PATTERN
)


# ============================================================
# FIND FLOOD ZIP MATCHING README DATE
# ============================================================

# Example:
# README = 15-Aug-2026
#
# Expected flood name:
# as_2026_15_08_1800.zip
#
year_text = EVENT_DATE.strftime("%Y")
day_text = EVENT_DATE.strftime("%d")
month_text = EVENT_DATE.strftime("%m")

flood_pattern = os.path.join(
    FLOOD_DIR,
    f"as_{year_text}_{day_text}_{month_text}_*.zip"
)

zip_files = sorted(
    glob.glob(
        flood_pattern
    )
)

if not zip_files:

    raise FileNotFoundError(
        "\nNo flood ZIP matching the README date was found.\n"
        f"README date : {EVENT_DATE.strftime('%d-%b-%Y')}\n"
        f"Search      : {flood_pattern}"
    )


# If several products exist for the same date,
# use the latest acquisition/time product.
FLOOD_ZIP = zip_files[-1]

print("\nFlood ZIP selected from README date:")
print(FLOOD_ZIP)


# ============================================================
# EXTRACT CORRECT FLOOD ZIP
# ============================================================

EXTRACT_DIR = os.path.join(
    FLOOD_DIR,
    "extracted",
    EVENT_DATE.strftime("%Y%m%d")
)

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)

with zipfile.ZipFile(
    FLOOD_ZIP,
    "r"
) as zf:

    zf.extractall(
        EXTRACT_DIR
    )

print("\nFlood ZIP extracted to:")
print(EXTRACT_DIR)


# ============================================================
# FIND SHAPEFILE FROM THIS EXTRACTION ONLY
# ============================================================

shp_files = glob.glob(
    os.path.join(
        EXTRACT_DIR,
        "**",
        "*.shp"
    ),
    recursive=True
)

if not shp_files:

    raise FileNotFoundError(
        f"No shapefile found inside:\n{FLOOD_ZIP}"
    )

if len(shp_files) > 1:

    print("\nMultiple shapefiles found:")

    for shp in shp_files:
        print("  ", shp)


FLOOD_SHP = sorted(
    shp_files
)[-1]

print("\nFlood shapefile selected:")
print(FLOOD_SHP)

# ============================================================
# EXTRACT ZIP
# ============================================================

EXTRACT_DIR = os.path.join(
    FLOOD_DIR,
    "extracted"
)

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)

with zipfile.ZipFile(
    FLOOD_ZIP,
    "r"
) as zf:

    zf.extractall(
        EXTRACT_DIR
    )

print("\nFlood ZIP extracted to:")
print(EXTRACT_DIR)

##########################################
#Extract Date
##########################################
def get_date_from_readme(readme_file):
    """
    Read acquisition date from README.txt and return it
    in DD-MM-YYYY format.

    Example README entry:
    Near Real Time Inundation Mapping :
    12-Aug-2026 (RISAT MRS SAR, 1800 Hrs)

    Returns:
        12-08-2026
    """

    if not os.path.exists(readme_file):
        raise FileNotFoundError(
            f"README file not found:\n{readme_file}"
        )

    with open(
        readme_file,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:
        content = f.read()

    pattern = (
        r"Near\s+Real\s+Time\s+Inundation\s+Mapping\s*:\s*"
        r"(\d{1,2}-[A-Za-z]{3}-\d{4})"
    )

    match = re.search(
        pattern,
        content,
        flags=re.IGNORECASE
    )

    if not match:
        raise RuntimeError(
            "Could not extract acquisition date from README.\n"
            "Expected an entry similar to:\n"
            "Near Real Time Inundation Mapping : "
            "12-Aug-2026 (RISAT MRS SAR, 1800 Hrs)"
        )

    readme_date = match.group(1).strip()

    try:
        dt = datetime.strptime(
            readme_date,
            "%d-%b-%Y"
        )
    except ValueError:
        raise RuntimeError(
            f"Invalid acquisition date in README: {readme_date}"
        )

    date_for_processing = dt.strftime("%d-%m-%Y")

    print("\n" + "=" * 70)
    print("DATE AUTOMATICALLY READ FROM README")
    print("=" * 70)
    print("README date     :", readme_date)
    print("Processing date :", date_for_processing)

    return date_for_processing

######################################################
### REad Date from README_FILE########
#####################################################

DATE = get_date_from_readme(README_PATTERN)

# Output root
OUTPUT_ROOT = os.path.join(WINDOWS_REPORT_ROOT, DATE)

# ============================================================
# SATELLITE IMAGE SOURCES
# ============================================================

# IMPORTANT:
# This is the ORIGINAL pre-downloaded / pre-flood satellite-image folder.
# These images are used ONLY in the district three-map report panel:
#
#       Pre-flood satellite image | Inundation map
#       LULC map                  | Inundation map
#
SATELLITE_DIR = os.path.join(
    WINDOWS_DATA_ROOT,
    "Assam_satellite"
)

# Current-event SAR imagery, STAC footprints and coverage maps generated
# automatically from Sentinel-1 / EOS-04 are stored separately here.
# These products are used ONLY for:
#   1. Overall Assam SAR coverage page
#   2. District-wise SAR coverage page
SAR_PRODUCTS_DIR = os.path.join(
    OUTPUT_ROOT,
    "sar_coverage_products"
)

AUTO_GENERATE_SATELLITE_COVERAGE = True
FOOTPRINT_REQUIRED = True

# CDSE Sentinel-1 catalogue
CDSE_STAC_ROOT = "https://stac.dataspace.copernicus.eu/v1"
CDSE_STAC_SEARCH_URL = f"{CDSE_STAC_ROOT}/search"
CDSE_STAC_COLLECTION = "sentinel-1-grd"

# Bhoonidhi catalogue
BHOONIDHI_TOKEN_URL = "https://bhoonidhi-api.nrsc.gov.in/auth/token"
BHOONIDHI_SEARCH_URL = "https://bhoonidhi-api.nrsc.gov.in/data/search"

# Map/display controls
FOOTPRINT_BASEMAP_SOURCE = (ctx.providers.OpenStreetMap.DE if HAVE_CONTEXTILY else None)
FOOTPRINT_BASEMAP_ZOOM = 8
FOOTPRINT_COLOR = "yellow"
FOOTPRINT_DISTRICT_COLOR = "red"
FOOTPRINT_ASSAM_COLOR = "#1f77b4"
SAR_ALPHA_DISTRICT = 0.55
SAR_ALPHA_OVERALL = 0.80
FOOTPRINT_CANVAS_WIDTH = 1400

# Optional credentials can be placed in environment variables so the
# complete run needs no keyboard input:
#   BHOONIDHI_USER_ID
#   BHOONIDHI_PASSWORD
#   CDSE_S3_ACCESS_KEY
#   CDSE_S3_SECRET_KEY


# Output products
INUNDATION_MAP_DIR = os.path.join(OUTPUT_ROOT, "inundation_maps")
OUTPUT_DOCX = os.path.join(OUTPUT_ROOT, f"RC_village_Report_{DATE}.docx")
OUTPUT_PDF = os.path.join(OUTPUT_ROOT, f"RC_village_Report_{DATE}.pdf")
OUTPUT_RC_EXCEL = os.path.join(OUTPUT_ROOT, f"RC_statistics_{DATE}.xlsx")
OUTPUT_VILLAGE_EXCEL = os.path.join(OUTPUT_ROOT, f"Village_statistics_{DATE}.xlsx")
OUTPUT_DISTRICT_EXCEL = os.path.join(OUTPUT_ROOT, f"District_summary_{DATE}.xlsx")

# Area CRS
AREA_CRS = "EPSG:32646"

# Attribute fields
DISTRICT_FIELD = "District_n"

RC_NAME_FIELD = "RC_NAME"
RC_DISTRICT_FIELD = "DISTRICT_N"
RC_AREA_FIELD = "RC_Area"

VILLAGE_NAME_FIELD = "villname"
VILLAGE_AREA_FIELD = "vill_area"

LULC_CLASS_FIELD = "DSCR1"

# LULC source categories
AGRICULTURE_SOURCE_NAME = "Agricultural Land"
BUILTUP_SOURCE_NAME = "Built Up"

# Optional sliver filter
MIN_VILLAGE_INUNDATED_AREA_HA = 0.0

# Ignore districts where the total satellite-derived inundated area is very
# small. This prevents tiny/sliver intersections at scene edges from being
# reported as flood-affected districts. Set to 0.0 to disable this filter.
MIN_DISTRICT_INUNDATED_AREA_HA = 100.0

# Word formatting
TABLE_HEADER_FILL = "9ACD32"
FONT_NAME = "Times New Roman"

# Number of village records targeted per half-page column.
# Adjust between 34-42 depending on font and row heights.
# Village table layout controls
# Physical-height splitting is used instead of a rigid row count.
VILLAGE_MAX_UNITS = 32.0
VILLAGE_FONT_SIZE = 8
VILLAGE_HEADER_FONT_SIZE = 8
RC_FONT_SIZE = 8

SATELLITE_SUFFIXES = [
    "_district.jpg", "_district.jpeg", "_district.png",
    ".jpg", ".jpeg", ".png"
]

LULC_SUFFIXES = [
    "_lulc.tif", "_lulc.tiff", "_lulc.png", "_lulc.jpg", "_lulc.jpeg"
]




# ============================================================
# TITLE PAGE + FINAL PDF HELPERS
# ============================================================

def _ordinal_suffix(day):
    day = int(day)
    if 10 <= day % 100 <= 20:
        return "th"
    return {1: "st", 2: "nd", 3: "rd"}.get(day % 10, "th")


def _build_report_serial(serial_input, year):
    """Accept a manual number (e.g. 28/0028) or a complete serial code."""
    value = str(serial_input).strip()

    if not value:
        raise ValueError("Report serial number cannot be blank.")

    if value.upper().startswith("NESAC-"):
        return value

    if value.isdigit():
        value = value.zfill(4)

    return f"NESAC-NERDRR-{year}-Flood-{value}"


def _xml_paragraph_text(paragraph, ns):
    return "".join(
        node.text or ""
        for node in paragraph.xpath(".//w:t", namespaces=ns)
    )


def _replace_xml_paragraph_text(paragraph, new_text, ns):
    """
    Replace the visible text of an OOXML paragraph while preserving the
    formatting of its first text run. This also works for text-box paragraphs.
    """
    text_nodes = paragraph.xpath(".//w:t", namespaces=ns)

    if not text_nodes:
        return False

    text_nodes[0].text = new_text

    for node in text_nodes[1:]:
        node.text = ""

    return True


def prepare_title_template(template_path, output_path, event_date, serial_input):
    """
    Create a customized copy of Title_new.docx.

    Only two values are changed:
      1. Flood scenario date -> README date
      2. Report serial number -> manually supplied value
    """
    if not os.path.exists(template_path):
        raise FileNotFoundError(
            f"Title template not found:\n{template_path}"
        )

    full_serial = _build_report_serial(
        serial_input,
        event_date.strftime("%Y")
    )

    title_text = (
        f"Flood scenario of Assam as on "
        f"{event_date.day}{_ordinal_suffix(event_date.day)} "
        f"{event_date.strftime('%B %Y')}"
    )

    print("\n" + "=" * 70)
    print("PREPARING TITLE PAGES")
    print("=" * 70)
    print("Title date :", title_text)
    print("Serial No. :", full_serial)

    # DOCX is a ZIP package. Modify only word/document.xml so all title-page
    # images, shapes, colours and other relationships remain unchanged.
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_dir = Path(temp_dir)
        package_dir = temp_dir / "package"

        with zipfile.ZipFile(template_path, "r") as zin:
            zin.extractall(package_dir)

        document_xml = package_dir / "word" / "document.xml"

        parser = etree.XMLParser(remove_blank_text=False)
        tree = etree.parse(str(document_xml), parser)
        root = tree.getroot()

        ns = {
            "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
        }

        date_count = 0
        serial_count = 0

        for paragraph in root.xpath(".//w:p", namespaces=ns):
            combined = _xml_paragraph_text(paragraph, ns)

            # Both page-1 and page-2 title dates are updated.
            if "Flood scenario of Assam" in combined and "as on" in combined:
                if _replace_xml_paragraph_text(paragraph, title_text, ns):
                    date_count += 1

            # Serial number is inside a text box in the supplied template.
            #if "NESAC-NERDRR-" in combined and "-Flood-" in combined:
              #  if _replace_xml_paragraph_text(paragraph, full_serial, ns):
                #    serial_count += 1
            if "NESAC-NERDRR-" in combined and "-Flood-" in combined:

                if _replace_xml_paragraph_text(
                    paragraph,
                    full_serial,
                    ns
                ):
            
                    serial_count += 1
            
                    # ==================================================
                    # CENTRE SERIAL NUMBER INSIDE ITS TEXT BOX
                    # ==================================================
            
                    pPr = paragraph.find(
                        "{http://schemas.openxmlformats.org/"
                        "wordprocessingml/2006/main}pPr"
                    )
            
                    if pPr is None:
            
                        pPr = etree.Element(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}pPr"
                        )
            
                        paragraph.insert(
                            0,
                            pPr
                        )
            
                    # Remove existing alignment if present
                    old_jc = pPr.find(
                        "{http://schemas.openxmlformats.org/"
                        "wordprocessingml/2006/main}jc"
                    )
            
                    if old_jc is not None:
                        pPr.remove(old_jc)
            
                    # Add centre alignment
                    jc = etree.SubElement(
                        pPr,
                        "{http://schemas.openxmlformats.org/"
                        "wordprocessingml/2006/main}jc"
                    )
            
                    jc.set(
                        "{http://schemas.openxmlformats.org/"
                        "wordprocessingml/2006/main}val",
                        "center"
                    )
            
                    # ==================================================
                    # FORMAT SERIAL NUMBER RUN
                    # ==================================================
            
                    runs = paragraph.xpath(
                        ".//w:r",
                        namespaces=ns
                    )
            
                    for run in runs:
            
                        # Ignore empty runs
                        text_value = "".join(
                            t.text or ""
                            for t in run.xpath(
                                ".//w:t",
                                namespaces=ns
                            )
                        )
            
                        if not text_value.strip():
                            continue
            
                        rPr = run.find(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}rPr"
                        )
            
                        if rPr is None:
            
                            rPr = etree.Element(
                                "{http://schemas.openxmlformats.org/"
                                "wordprocessingml/2006/main}rPr"
                            )
            
                            run.insert(
                                0,
                                rPr
                            )
            
                        # ----------------------------------------------
                        # FONT SIZE
                        # Word XML uses half-points.
                        # 24 = 12 pt
                        # ----------------------------------------------
            
                        sz = rPr.find(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}sz"
                        )
            
                        if sz is None:
            
                            sz = etree.SubElement(
                                rPr,
                                "{http://schemas.openxmlformats.org/"
                                "wordprocessingml/2006/main}sz"
                            )
            
                        sz.set(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}val",
                            "20"
                        )
            
                        # Also set complex-script size
                        szCs = rPr.find(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}szCs"
                        )
            
                        if szCs is None:
            
                            szCs = etree.SubElement(
                                rPr,
                                "{http://schemas.openxmlformats.org/"
                                "wordprocessingml/2006/main}szCs"
                            )
            
                        szCs.set(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}val",
                            "20"
                        )
            
                        # ----------------------------------------------
                        # FONT COLOUR - WHITE
                        # ----------------------------------------------
            
                        color = rPr.find(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}color"
                        )
            
                        if color is None:
            
                            color = etree.SubElement(
                                rPr,
                                "{http://schemas.openxmlformats.org/"
                                "wordprocessingml/2006/main}color"
                            )
            
                        color.set(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}val",
                            "FFFFFF"
                        )
            
                        # ----------------------------------------------
                        # BORDER AROUND THE TEXT ITSELF
                        # ----------------------------------------------
            
                        old_border = rPr.find(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}bdr"
                        )
            
                        if old_border is not None:
                            rPr.remove(old_border)
            
                        border = etree.SubElement(
                            rPr,
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}bdr"
                        )
            
                        border.set(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}val",
                            "single"
                        )
            
                        # Border thickness
                        border.set(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}sz",
                            "8"
                        )
            
                        # Space between text and border
                        border.set(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}space",
                            "4"
                        )
            
                        # White border on blue background
                        border.set(
                            "{http://schemas.openxmlformats.org/"
                            "wordprocessingml/2006/main}color",
                            "FFFFFF"
                        )

        if date_count == 0:
            raise RuntimeError(
                "Could not locate the flood-scenario date in Title_new.docx."
            )

        if serial_count == 0:
            raise RuntimeError(
                "Could not locate the report serial number in Title_new.docx."
            )

        tree.write(
            str(document_xml),
            xml_declaration=True,
            encoding="UTF-8",
            standalone="yes"
        )

        with zipfile.ZipFile(output_path, "w", zipfile.ZIP_DEFLATED) as zout:
            for file_path in package_dir.rglob("*"):
                if not file_path.is_file():
                    continue

                arcname = file_path.relative_to(package_dir)
                zout.write(
                    file_path,
                    str(arcname).replace("\\", "/")
                )

    print("Customized title pages:", output_path)
    return output_path, full_serial


def configure_generated_report_section(section):
    """
    Apply exactly the same landscape A4 settings used by the existing report
    to the new section that follows the two untouched title pages.
    """
    section.orientation = WD_ORIENT.LANDSCAPE
    section.page_width = Cm(29.7)
    section.page_height = Cm(21.0)

    section.top_margin = Cm(0.65)
    section.bottom_margin = Cm(0.65)
    section.left_margin = Cm(0.90)
    section.right_margin = Cm(0.90)

    section.header_distance = Cm(0.30)
    section.footer_distance = Cm(0.30)

    section.header.is_linked_to_previous = False
    section.footer.is_linked_to_previous = False


def restart_report_page_numbering(section, start=1):
    """Restart the generated report at page 1 after the two title pages."""
    sectPr = section._sectPr
    pgNumType = sectPr.find(qn("w:pgNumType"))

    if pgNumType is None:
        pgNumType = OxmlElement("w:pgNumType")
        sectPr.append(pgNumType)

    pgNumType.set(qn("w:start"), str(start))


def convert_docx_to_pdf(docx_path, pdf_path):
    """
    Convert the completed DOCX (including the two title pages) to PDF.

    Windows conversion priority:
      1. Microsoft Word via pywin32
      2. docx2pdf
      3. LibreOffice / soffice

    The first available method is used automatically.
    """
    docx_path = os.path.abspath(docx_path)
    pdf_path = os.path.abspath(pdf_path)

    print("\n" + "=" * 70)
    print("CONVERTING FINAL REPORT TO PDF")
    print("=" * 70)
    print("DOCX:", docx_path)
    print("PDF :", pdf_path)

    # --------------------------------------------------------
    # Method 1: Microsoft Word COM (best fidelity on Windows)
    # --------------------------------------------------------
    try:
        import win32com.client

        word = win32com.client.DispatchEx("Word.Application")
        word.Visible = False
        word.DisplayAlerts = 0

        document = None

        try:
            document = word.Documents.Open(docx_path)

            # 17 = wdFormatPDF
            document.SaveAs(pdf_path, FileFormat=17)

        finally:
            if document is not None:
                document.Close(False)
            word.Quit()

        if os.path.exists(pdf_path) and os.path.getsize(pdf_path) > 0:
            print("PDF conversion successful using Microsoft Word.")
            return pdf_path

    except Exception as exc:
        print("Microsoft Word PDF conversion unavailable:", exc)

    # --------------------------------------------------------
    # Method 2: docx2pdf (also uses Word on Windows)
    # --------------------------------------------------------
    try:
        from docx2pdf import convert
        convert(docx_path, pdf_path)

        if os.path.exists(pdf_path) and os.path.getsize(pdf_path) > 0:
            print("PDF conversion successful using docx2pdf.")
            return pdf_path

    except Exception as exc:
        print("docx2pdf conversion unavailable:", exc)

    # --------------------------------------------------------
    # Method 3: LibreOffice
    # --------------------------------------------------------
    libreoffice_candidates = [
        "soffice",
        "libreoffice",
        r"C:\\Program Files\\LibreOffice\\program\\soffice.exe",
        r"C:\\Program Files (x86)\\LibreOffice\\program\\soffice.exe",
    ]

    for executable in libreoffice_candidates:
        try:
            result = subprocess.run(
                [
                    executable,
                    "--headless",
                    "--convert-to",
                    "pdf",
                    "--outdir",
                    os.path.dirname(pdf_path),
                    docx_path,
                ],
                check=True,
                capture_output=True,
                text=True
            )

            generated_pdf = os.path.join(
                os.path.dirname(pdf_path),
                Path(docx_path).stem + ".pdf"
            )

            if os.path.exists(generated_pdf):
                if os.path.normcase(generated_pdf) != os.path.normcase(pdf_path):
                    if os.path.exists(pdf_path):
                        os.remove(pdf_path)
                    os.replace(generated_pdf, pdf_path)

            if os.path.exists(pdf_path) and os.path.getsize(pdf_path) > 0:
                print("PDF conversion successful using LibreOffice.")
                return pdf_path

        except Exception:
            continue

    raise RuntimeError(
        "Could not convert the final DOCX to PDF. Install pywin32/Word, "
        "docx2pdf, or LibreOffice. The DOCX itself has been saved successfully."
    )


# ============================================================
# AUTOMATIC SATELLITE FOOTPRINT / SCENE-COVERAGE GENERATION
# ============================================================

def _fp_safe_filename(text):
    text = str(text).strip()
    text = re.sub(r'[<>:"/\\|?*]+', "_", text)
    text = re.sub(r"\s+", "_", text)
    return text


def _fp_detect_product_platform(product_id, props):
    product_id_upper = str(product_id).upper()

    if (
        product_id_upper.startswith("E04_")
        or "EOS-04" in product_id_upper
        or "EOS04" in product_id_upper
    ):
        return "EOS-04"

    for platform in ["S1A", "S1B", "S1C", "S1D"]:
        if product_id_upper.startswith(platform + "_") or platform in product_id_upper:
            return platform

    candidates = [
        props.get("Platform"),
        props.get("platform"),
        props.get("Satellite"),
        props.get("satellite"),
        props.get("Mission"),
        props.get("mission"),
        props.get("constellation"),
    ]

    for value in candidates:
        if not value:
            continue

        clean = (
            str(value).upper()
            .replace("-", "")
            .replace("_", "")
            .replace(" ", "")
        )

        if "EOS04" in clean or "RISAT" in clean:
            return "EOS-04"

        for platform in ["S1A", "S1B", "S1C", "S1D"]:
            if clean in [platform, platform.replace("S1", "SENTINEL1")]:
                return platform

    return None


def _fp_interpret_satellite(text):
    text_upper = str(text).upper().strip()
    text_clean = text_upper.replace("_", " ").replace("-", " ")
    text_clean = " ".join(text_clean.split())

    platform = None
    mode = None
    sensor = None

    if (
        "RISAT" in text_clean
        or "EOS 04" in text_clean
        or "EOS04" in text_upper
        or "E04" in text_upper
    ):
        platform = "EOS-04"
        sensor = "SAR"

        if "MRS" in text_clean:
            mode = "MRS"
        elif "FRS1" in text_clean or "FRS 1" in text_clean:
            mode = "FRS1"
        elif "FRS2" in text_clean or "FRS 2" in text_clean:
            mode = "FRS2"
        elif "CRS" in text_clean:
            mode = "CRS"
        elif "SM" in text_clean:
            mode = "SM"

    else:
        for p in ["S1A", "S1B", "S1C", "S1D"]:
            long_name = "SENTINEL 1" + p[-1]
            if p in text_clean or long_name in text_clean:
                platform = p
                sensor = "SAR"
                break

    return platform, mode, sensor


def _fp_choose_source(platform, mode):
    if platform == "EOS-04":
        collections = {
            "MRS": "EOS-04_SAR-MRS_L2B",
            "FRS1": "EOS-04_SAR-FRS1_L2A",
            "FRS2": "EOS-04_SAR-FRS2_L2A",
            "CRS": "EOS-04_SAR-CRS_L2B",
            "SM": "EOS-04_SAR-MRS_SM",
        }
        if mode not in collections:
            raise ValueError(
                "EOS-04/RISAT detected in README, but acquisition mode "
                "could not be identified."
            )
        return "BHOONIDHI", collections[mode]

    if platform in ["S1A", "S1B", "S1C", "S1D"]:
        return "CDSE", CDSE_STAC_COLLECTION

    raise ValueError(f"Unsupported README satellite/platform: {platform}")


def _fp_read_latest_nrt_entry(readme_file):
    with open(readme_file, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    pattern = (
        r"Near\s+Real\s+Time\s+Inundation\s+Mapping\s*:\s*"
        r"(\d{1,2}-[A-Za-z]{3}-\d{4})"
        r"\s*\("
        r"([^,]+)"
        r",\s*"
        r"(\d{4})\s*Hrs"
        r"\)"
    )

    matches = list(re.finditer(pattern, content, flags=re.IGNORECASE))

    if not matches:
        raise RuntimeError(
            "Could not find a complete Near Real Time Inundation Mapping "
            "entry in README."
        )

    records = []
    for match in matches:
        dt = datetime.strptime(match.group(1), "%d-%b-%Y")
        hhmm = match.group(3)
        dt = dt.replace(hour=int(hhmm[:2]), minute=int(hhmm[2:]))

        records.append({
            "date_text": match.group(1),
            "satellite_text": match.group(2).strip(),
            "time_text": hhmm,
            "datetime": dt,
        })

    return sorted(records, key=lambda x: x["datetime"], reverse=True)[0]


def _fp_find_quicklook_asset(assets):
    if not isinstance(assets, dict):
        return None, None

    for key in ["thumbnail", "preview", "quicklook", "overview", "browse"]:
        asset = assets.get(key)
        if isinstance(asset, dict) and asset.get("href"):
            return key, asset

    for key, asset in assets.items():
        if not isinstance(asset, dict):
            continue
        href = str(asset.get("href", ""))
        media = str(asset.get("type", "")).lower()
        if (
            href.lower().split("?")[0].endswith((".jpg", ".jpeg", ".png"))
            or media.startswith("image/jpeg")
            or media.startswith("image/png")
        ):
            return key, asset

    return None, None


def _fp_asset_looks_like_tiff(key, asset):
    if not isinstance(asset, dict):
        return False

    href = str(asset.get("href", ""))
    media = str(asset.get("type", "")).lower()
    title = str(asset.get("title", "")).lower()
    key_l = str(key).lower()

    return (
        href.lower().split("?")[0].endswith((".tif", ".tiff"))
        or "geotiff" in media
        or "image/tiff" in media
        or "cloud-optimized" in media
        or "cog" in title
        or "cog" in key_l
    )


def _fp_find_sentinel_cog_asset(assets):
    if not isinstance(assets, dict):
        return None, None

    for pol in ["vv", "vh", "hh", "hv"]:
        for key, asset in assets.items():
            if not isinstance(asset, dict):
                continue

            key_l = str(key).lower()
            title_l = str(asset.get("title", "")).lower()
            roles = [str(r).lower() for r in asset.get("roles", [])]

            if any(r in roles for r in ["thumbnail", "overview"]):
                continue

            is_pol = (
                key_l == pol
                or key_l.endswith("_" + pol)
                or key_l.startswith(pol + "_")
                or f"-{pol}-" in title_l
                or f"_{pol}_" in title_l
            )

            if is_pol and _fp_asset_looks_like_tiff(key, asset):
                return key, asset

    for key, asset in assets.items():
        roles = [str(r).lower() for r in asset.get("roles", [])] if isinstance(asset, dict) else []
        if any(r in roles for r in ["thumbnail", "overview"]):
            continue
        if _fp_asset_looks_like_tiff(key, asset):
            return key, asset

    return None, None


def _fp_stretch_sar(arr):
    data = np.ma.asarray(arr, dtype=np.float32)

    if data.count() == 0:
        raise ValueError("SAR raster contains no valid pixels.")

    values = data.compressed()
    values = values[np.isfinite(values)]

    if values.size == 0:
        raise ValueError("SAR raster contains no finite pixels.")

    work = data.copy()

    if np.nanmin(values) >= 0:
        work = np.ma.maximum(work, 0)
        # np.log1p works with MaskedArray and avoids np.ma.log1p
        # compatibility problems.
        work = np.log1p(work)

    finite = work.compressed()
    finite = finite[np.isfinite(finite)]

    p2, p98 = np.nanpercentile(finite, [2, 98])

    if not np.isfinite(p2) or not np.isfinite(p98) or p98 <= p2:
        p2 = float(np.nanmin(finite))
        p98 = float(np.nanmax(finite))

    if p98 <= p2:
        p98 = p2 + 1.0

    stretched = np.ma.clip((work - p2) / (p98 - p2), 0, 1)
    image = stretched.filled(0).astype(np.float32) * 255.0

    alpha = (~np.ma.getmaskarray(data)).astype(np.float32)
    alpha[~np.isfinite(np.asarray(data.filled(np.nan)))] = 0.0

    return image, alpha


def _fp_get_cdse_credentials():
    """
    Return CDSE S3 credentials.

    Priority:
      1. CDSE_S3_ACCESS_KEY / CDSE_S3_SECRET_KEY environment variables
      2. values already entered during this Python run
      3. interactive prompt

    The prompt therefore appears only once.
    """
    global CDSE_S3_ACCESS_KEY, CDSE_S3_SECRET_KEY

    env_access = os.environ.get("CDSE_S3_ACCESS_KEY", "").strip()
    env_secret = os.environ.get("CDSE_S3_SECRET_KEY", "").strip()

    if env_access and env_secret:
        return env_access, env_secret

    if CDSE_S3_ACCESS_KEY and CDSE_S3_SECRET_KEY:
        return CDSE_S3_ACCESS_KEY, CDSE_S3_SECRET_KEY

    print("\nCDSE Sentinel-1 COG access requires S3 credentials.")

    CDSE_S3_ACCESS_KEY = input(
        "Enter CDSE S3 Access Key: "
    ).strip()

    CDSE_S3_SECRET_KEY = getpass.getpass(
        "Enter CDSE S3 Secret Key: "
    ).strip()

    if not CDSE_S3_ACCESS_KEY or not CDSE_S3_SECRET_KEY:
        raise RuntimeError(
            "CDSE S3 Access Key and Secret Key are required "
            "because USE_SENTINEL_SAR_RASTER=True."
        )

    return CDSE_S3_ACCESS_KEY, CDSE_S3_SECRET_KEY


def _fp_prepare_sentinel_cog(asset):
    """
    Read the real Sentinel-1 measurement COG through CDSE S3 and warp it
    to EPSG:3857 for plotting over OpenStreetMap.

    Sentinel-1D COGs may have src.crs=None but contain GCPs in EPSG:4326.
    WarpedVRT/GDAL can use those GCPs directly, so src.crs=None is not
    treated as an error.
    """
    if not USE_SENTINEL_SAR_RASTER:
        return None

    if not HAVE_SENTINEL_COG_SUPPORT:
        raise RuntimeError(
            "Sentinel COG support is unavailable. Install rasterio and boto3."
        )

    access_key, secret_key = _fp_get_cdse_credentials()

    href = str(asset.get("href", "")).strip()

    if not href:
        raise ValueError(
            "Selected Sentinel georaster asset has no href."
        )

    print("Raster URL:", href)

    boto_session = boto3.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name="default",
    )

    aws_session = AWSSession(
        boto_session,
        endpoint_url="eodata.dataspace.copernicus.eu",
    )

    rasterio_env = {
        "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
        "CPL_VSIL_CURL_ALLOWED_EXTENSIONS":
            ".tif,.tiff,.TIF,.TIFF",
        "VSI_CACHE": "TRUE",
        "VSI_CACHE_SIZE": "50000000",
        "AWS_S3_ENDPOINT":
            "eodata.dataspace.copernicus.eu",
        "AWS_HTTPS": "YES",
        "AWS_VIRTUAL_HOSTING": "FALSE",
    }

    with rasterio.Env(
        session=aws_session,
        **rasterio_env
    ):
        with rasterio.open(href) as src:

            print("Source CRS       :", src.crs)
            print("Source size      :", src.width, "x", src.height)
            print("Source bands     :", src.count)

            gcps, gcp_crs = src.gcps

            print("Number of GCPs   :", len(gcps))
            print("GCP CRS          :", gcp_crs)

            if gcps:
                print("First GCP        :", gcps[0])

            with WarpedVRT(
                src,
                crs="EPSG:3857",
                resampling=Resampling.bilinear,
            ) as vrt:

                if vrt.width <= 0 or vrt.height <= 0:
                    raise ValueError(
                        "Invalid warped Sentinel raster dimensions."
                    )

                scale = min(
                    1.0,
                    FOOTPRINT_CANVAS_WIDTH / float(vrt.width)
                )

                out_width = max(
                    1,
                    int(round(vrt.width * scale))
                )

                out_height = max(
                    1,
                    int(round(vrt.height * scale))
                )

                if out_height > 5000:
                    extra_scale = (
                        5000.0 / float(out_height)
                    )

                    out_height = 5000
                    out_width = max(
                        1,
                        int(round(out_width * extra_scale))
                    )

                print(
                    "Reading display raster:",
                    out_width,
                    "x",
                    out_height
                )

                arr = vrt.read(
                    1,
                    out_shape=(
                        out_height,
                        out_width
                    ),
                    masked=True,
                    resampling=Resampling.bilinear,
                )

                bounds = vrt.bounds

    # _fp_stretch_sar uses np.log1p, NOT np.ma.log1p.
    image, alpha = _fp_stretch_sar(arr)

    extent = [
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top,
    ]

    print(
        "Display size     :",
        image.shape[1],
        "x",
        image.shape[0]
    )
    print("Raster extent    :", extent)

    return image, alpha, extent


def _fp_search_scenes(event, search_geom):
    platform, mode, sensor = _fp_interpret_satellite(
        event["satellite_text"]
    )

    source, collection = _fp_choose_source(platform, mode)

    start = event["datetime"].replace(
        hour=0, minute=0, second=0, microsecond=0
    )
    end = event["datetime"].replace(
        hour=23, minute=59, second=59, microsecond=0
    )

    datetime_query = (
        start.strftime("%Y-%m-%dT%H:%M:%SZ")
        + "/"
        + end.strftime("%Y-%m-%dT%H:%M:%SZ")
    )

    print("\n" + "=" * 70)
    print("AUTOMATIC SATELLITE FOOTPRINT SEARCH")
    print("=" * 70)
    print("README satellite :", event["satellite_text"])
    print("Platform         :", platform)
    print("Source           :", source)
    print("Collection       :", collection)
    print("Search period    :", datetime_query)

    if source == "CDSE":
        payload = {
            "collections": [collection],
            "datetime": datetime_query,
            "intersects": search_geom.__geo_interface__,
            "limit": 500,
        }

        response = requests.post(
            CDSE_STAC_SEARCH_URL,
            json=payload,
            headers={"Accept": "application/geo+json, application/json"},
            timeout=120,
        )
        response.raise_for_status()

        try:
            results = response.json()
        except Exception:
            minx, miny, maxx, maxy = search_geom.bounds

            response = requests.get(
                CDSE_STAC_SEARCH_URL,
                params={
                    "collections": collection,
                    "datetime": datetime_query,
                    "bbox": f"{minx},{miny},{maxx},{maxy}",
                    "limit": 500,
                },
                headers={"Accept": "application/geo+json, application/json"},
                timeout=120,
            )
            response.raise_for_status()
            results = response.json()

        features = results.get("features", [])

        cleaned = []
        for item in features:
            geom_json = item.get("geometry")
            if not geom_json:
                continue

            try:
                geom = shape(geom_json)
            except Exception:
                continue

            if not geom.intersects(search_geom):
                continue

            product_id = str(item.get("id", "")).upper()
            props = item.get("properties", {})
            detected = _fp_detect_product_platform(product_id, props)

            if detected != platform:
                continue

            mode_value = str(props.get("sar:instrument_mode", "")).upper()
            is_iw = mode_value == "IW" or "_IW_" in product_id

            product_type = " ".join(
                str(v).upper()
                for v in [
                    props.get("product:type"),
                    props.get("product_type"),
                    props.get("processing:level"),
                ]
                if v is not None
            )
            is_grd = "GRD" in product_type or "_GRD" in product_id

            if is_iw and is_grd:
                cleaned.append(item)

        features = cleaned

    else:
        user_id = os.environ.get("BHOONIDHI_USER_ID", "").strip()
        password = os.environ.get("BHOONIDHI_PASSWORD", "").strip()

        if not user_id:
            user_id = input("Enter Bhoonidhi User ID: ").strip()
        if not password:
            password = getpass.getpass("Enter Bhoonidhi Password: ")

        login = requests.post(
            BHOONIDHI_TOKEN_URL,
            json={
                "userId": user_id,
                "password": password,
                "grant_type": "password",
            },
            timeout=60,
        )
        login.raise_for_status()

        token = login.json()["access_token"]

        response = requests.post(
            BHOONIDHI_SEARCH_URL,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json",
            },
            json={
                "collections": [collection],
                "datetime": datetime_query,
                "intersects": search_geom.__geo_interface__,
                "limit": 500,
            },
            timeout=120,
        )
        response.raise_for_status()
        features = response.json().get("features", [])

    print("Matching scenes found:", len(features))

    if not features:
        raise RuntimeError(
            f"No suitable {platform} scenes found for "
            f"{event['date_text']} over Assam."
        )

    return features, platform, mode, source



# ============================================================
# EOS-04 / BHOONIDHI QUICKLOOK GEOREFERENCING HELPERS
# ============================================================

EOS04_VALID_THRESHOLD = 3

_EOS04_TO_WEB = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True
)


def _fp_order_geo_corners(points):
    points = np.asarray(points, dtype=np.float64)

    if points.shape != (4, 2):
        raise ValueError("Exactly four geographic corners are required.")

    order_y = np.argsort(points[:, 1])
    bottom = points[order_y[:2]]
    top = points[order_y[2:]]

    top = top[np.argsort(top[:, 0])]
    bottom = bottom[np.argsort(bottom[:, 0])]

    top_left = top[0]
    top_right = top[1]
    bottom_left = bottom[0]
    bottom_right = bottom[1]

    return np.array(
        [top_left, top_right, bottom_right, bottom_left],
        dtype=np.float64
    )


def _fp_scene_four_corners(scene_geom):
    geom = scene_geom

    if geom.geom_type == "MultiPolygon":
        geom = max(geom.geoms, key=lambda g: g.area)

    if geom.geom_type != "Polygon":
        raise ValueError(
            f"Unsupported EOS-04 footprint geometry: {geom.geom_type}"
        )

    coords = list(geom.exterior.coords)[:-1]

    if len(coords) != 4:
        rectangle = geom.minimum_rotated_rectangle
        coords = list(rectangle.exterior.coords)[:-1]

    if len(coords) != 4:
        raise ValueError(
            "Could not derive four EOS-04 footprint corners."
        )

    return _fp_order_geo_corners(coords)


def _fp_detect_quicklook_corners(img_array):
    img_array = np.asarray(img_array, dtype=np.float32)

    valid = img_array > EOS04_VALID_THRESHOLD
    yy, xx = np.where(valid)

    if len(xx) < 10:
        raise ValueError(
            "Could not detect valid SAR area inside EOS-04 quicklook."
        )

    points = np.column_stack([xx, yy]).astype(np.float64)

    sums = points[:, 0] + points[:, 1]
    diffs = points[:, 0] - points[:, 1]

    top_left = points[np.argmin(sums)]
    top_right = points[np.argmax(diffs)]
    bottom_right = points[np.argmax(sums)]
    bottom_left = points[np.argmin(diffs)]

    return np.array(
        [top_left, top_right, bottom_right, bottom_left],
        dtype=np.float32
    )


def _fp_prepare_eos04_quicklook(quicklook_asset, scene_geom):
    """
    Download the EOS-04 Bhoonidhi JPG/PNG quicklook and projectively
    warp it to the authoritative scene footprint in EPSG:3857.
    """

    href = str(quicklook_asset.get("href", "")).strip()

    if not href:
        raise ValueError("EOS-04 quicklook asset has no href.")

    print("EOS-04 quicklook URL:", href)

    response = requests.get(href, timeout=120)
    response.raise_for_status()

    img = Image.open(BytesIO(response.content)).convert("L")
    img_array = np.asarray(img, dtype=np.float32)

    h, w = img_array.shape

    print("EOS-04 browse size:", w, "x", h)

    src_corners = _fp_detect_quicklook_corners(img_array)

    original_mask = np.zeros((h, w), dtype=np.float32)
    rr, cc = polygon(
        src_corners[:, 1],
        src_corners[:, 0],
        shape=(h, w)
    )
    original_mask[rr, cc] = 1.0

    corners_lonlat = _fp_scene_four_corners(scene_geom)

    corners_xy = []
    for lon, lat in corners_lonlat:
        x, y = _EOS04_TO_WEB.transform(lon, lat)
        corners_xy.append([x, y])

    corners_xy = np.asarray(corners_xy, dtype=np.float64)

    xmin = float(corners_xy[:, 0].min())
    xmax = float(corners_xy[:, 0].max())
    ymin = float(corners_xy[:, 1].min())
    ymax = float(corners_xy[:, 1].max())

    map_width = xmax - xmin
    map_height = ymax - ymin

    if map_width <= 0 or map_height <= 0:
        raise ValueError("Invalid EOS-04 map extent.")

    canvas_width = FOOTPRINT_CANVAS_WIDTH
    canvas_height = max(
        100,
        int(round(canvas_width * map_height / map_width))
    )

    if canvas_height > 5000:
        extra_scale = 5000.0 / float(canvas_height)
        canvas_height = 5000
        canvas_width = max(
            1,
            int(round(canvas_width * extra_scale))
        )

    dst_corners = []

    for x, y in corners_xy:
        px = ((x - xmin) / map_width) * (canvas_width - 1)
        py = ((ymax - y) / map_height) * (canvas_height - 1)
        dst_corners.append([px, py])

    dst_corners = np.asarray(dst_corners, dtype=np.float32)

    transform = ProjectiveTransform()
    success = transform.estimate(src_corners, dst_corners)

    if not success:
        raise ValueError(
            "EOS-04 projective transformation failed."
        )

    warped = warp(
        img_array,
        inverse_map=transform.inverse,
        output_shape=(canvas_height, canvas_width),
        preserve_range=True,
        order=1,
        mode="constant",
        cval=0
    ).astype(np.float32)

    warped_mask = warp(
        original_mask,
        inverse_map=transform.inverse,
        output_shape=(canvas_height, canvas_width),
        preserve_range=True,
        order=0,
        mode="constant",
        cval=0
    ).astype(np.float32)

    warped_mask = np.clip(warped_mask, 0.0, 1.0)

    extent = [xmin, xmax, ymin, ymax]

    print(
        "EOS-04 display size:",
        canvas_width,
        "x",
        canvas_height
    )
    print("EOS-04 raster extent:", extent)

    return warped, warped_mask, extent



def _fp_prepare_scene(item, source):
    product_id = str(item.get("id", "Unknown"))
    props = item.get("properties", {})
    geom_json = item.get("geometry")

    if not geom_json:
        return None

    scene_geom = shape(geom_json)

    scene_gdf = gpd.GeoDataFrame(
        {"Product_ID": [product_id]},
        geometry=[scene_geom],
        crs="EPSG:4326",
    )
    scene_web = scene_gdf.to_crs("EPSG:3857")

    result = {
        "product_id": product_id,
        "image": None,
        "alpha": None,
        "extent": None,
        "scene_web": scene_web,
    }

    assets = item.get("assets", {})

    # Sentinel: use actual COG if S3 credentials have been configured.
    if source == "CDSE":
        _, cog_asset = _fp_find_sentinel_cog_asset(assets)

        if cog_asset is not None:
            try:
                prepared = _fp_prepare_sentinel_cog(cog_asset)
                if prepared is not None:
                    image, alpha, extent = prepared
                    result["image"] = image
                    result["alpha"] = alpha
                    result["extent"] = extent
            except Exception as exc:
                print(
                    "Sentinel COG display warning:",
                    product_id,
                    exc
                )

        return result

    # ========================================================
    # EOS-04 / RISAT-1A QUICKLOOK
    # ========================================================

    quicklook_key, quicklook_asset = _fp_find_quicklook_asset(assets)

    if quicklook_asset is None:
        print("EOS-04 quicklook not found for:", product_id)
        print("Available asset keys:", list(assets.keys()))
        print("Using EOS-04 footprint only.")
        return result

    print("Using EOS-04 quicklook asset:", quicklook_key)

    try:
        image, alpha, extent = _fp_prepare_eos04_quicklook(
            quicklook_asset,
            scene_geom
        )

        # IMPORTANT:
        # Use the SAME fields as Sentinel so _fp_draw_scenes()
        # actually draws the EOS-04 image.
        result["image"] = image
        result["alpha"] = alpha
        result["extent"] = extent

        print(
            "EOS-04 quicklook georeferenced successfully:",
            product_id
        )

    except Exception as exc:
        print(
            "EOS-04 quicklook could not be georeferenced:",
            product_id
        )
        print("Reason:", exc)
        print("Using EOS-04 footprint only.")

    return result


def _fp_add_basemap(ax):
    """Add OSM only when requested; a tile failure does not stop the report."""
    if not USE_OSM_BASEMAP:
        return

    if not HAVE_CONTEXTILY or FOOTPRINT_BASEMAP_SOURCE is None:
        print("OSM basemap skipped: contextily is not available.")
        return

    try:
        ctx.add_basemap(
            ax,
            source=FOOTPRINT_BASEMAP_SOURCE,
            zoom=FOOTPRINT_BASEMAP_ZOOM,
            zorder=1,
        )
    except Exception as exc:
        print("OSM basemap warning:", exc)


def _fp_draw_scenes(
    ax,
    scenes,
    alpha_value,
    draw_footprints=True
):
    """
    Draw actual SAR raster first, then optional STAC footprints.
    """
    for scene in scenes:

        if (
            scene.get("image") is not None
            and scene.get("alpha") is not None
            and scene.get("extent") is not None
        ):
            ax.imshow(
                scene["image"],
                extent=scene["extent"],
                origin="upper",
                cmap="gray",
                alpha=scene["alpha"] * alpha_value,
                interpolation="bilinear",
                zorder=3,
            )

        if draw_footprints:
            scene["scene_web"].boundary.plot(
                ax=ax,
                edgecolor=FOOTPRINT_COLOR,
                linewidth=1.7,
                zorder=5,
            )


def generate_satellite_coverage_maps(
    dist_gdf,
    target_districts
):
    """
    Create all satellite products required by the report.

    Products
    --------
    1. Overall Assam SAR + footprint + OSM coverage map.
    2. One district-wise SAR + footprint + OSM coverage map per
       flood-affected district.
    3. One optional clean district SAR view for diagnostic/reference use.
       It is NOT inserted into the original district three-map report panel.

    Returns
    -------
    overall_path : str
    coverage_lookup : dict
        District -> district coverage image.
    sar_lookup : dict
        District -> clean SAR image used inside the district report panel.
    """

    ensure_dir(SAR_PRODUCTS_DIR)

    event = _fp_read_latest_nrt_entry(
        README_PATTERN
    )

    assam_ll = (
        dist_gdf
        .to_crs("EPSG:4326")
        .copy()
    )

    try:
        search_geom = assam_ll.geometry.union_all()
    except AttributeError:
        search_geom = assam_ll.geometry.unary_union

    search_geom = search_geom.simplify(
        0.001,
        preserve_topology=True
    )

    if not search_geom.is_valid:
        search_geom = search_geom.buffer(0)

    features, platform, mode, source = (
        _fp_search_scenes(
            event,
            search_geom
        )
    )

    scenes = []

    for item in features:

        try:
            prepared = _fp_prepare_scene(
                item,
                source
            )

            if prepared is not None:
                scenes.append(
                    prepared
                )

        except Exception as exc:
            print(
                "Scene preparation warning:",
                item.get("id", ""),
                exc
            )

    if not scenes:
        raise RuntimeError(
            "No satellite scene could be prepared."
        )

    # On Windows, if Sentinel was requested and real raster display is enabled,
    # fail clearly rather than silently producing footprint-only maps.
    if (
        source == "CDSE"
        and USE_SENTINEL_SAR_RASTER
        and not any(
            scene.get("image") is not None
            for scene in scenes
        )
    ):
        raise RuntimeError(
            "Sentinel-1 scenes were found, but none of the actual COG "
            "rasters could be read. Check CDSE S3 credentials, rasterio/GDAL "
            "and Internet access."
        )

    if (
        source == "BHOONIDHI"
        and not any(
            scene.get("image") is not None
            for scene in scenes
        )
    ):
        raise RuntimeError(
            "EOS-04 scenes were found in Bhoonidhi, but none of the "
            "quicklook/browse images could be displayed. Check the printed "
            "EOS-04 asset keys, quicklook URL and Bhoonidhi access."
        )

    assam_web = (
        assam_ll
        .to_crs("EPSG:3857")
    )

    # --------------------------------------------------------
    # Coverage statistics
    # --------------------------------------------------------

    assam_area = (
        assam_ll
        .to_crs(AREA_CRS)
    )

    scene_geometries_area = [
        scene["scene_web"]
        .to_crs(AREA_CRS)
        .geometry
        .iloc[0]
        for scene in scenes
    ]

    footprint_union_area = unary_union(
        scene_geometries_area
    )

    coverage_records = []

    for _, row in assam_area.iterrows():

        district_name = (
            str(row[DISTRICT_FIELD])
            .strip()
            .upper()
        )

        district_geom = row.geometry

        if (
            district_geom is None
            or district_geom.is_empty
        ):
            continue

        district_area_m2 = (
            district_geom.area
        )

        intersection = (
            district_geom
            .intersection(
                footprint_union_area
            )
        )

        covered_area_m2 = (
            0.0
            if intersection.is_empty
            else intersection.area
        )

        coverage_percent = (
            covered_area_m2
            / district_area_m2
            * 100.0
            if district_area_m2 > 0
            else 0.0
        )

        coverage_records.append({
            "District":
                district_name,

            "District_Area_km2":
                district_area_m2 / 1e6,

            "Covered_Area_km2":
                covered_area_m2 / 1e6,

            "Coverage_Percent":
                coverage_percent,
        })

    coverage_df = pd.DataFrame(
        coverage_records
    )

    if not coverage_df.empty:

        for col in [
            "District_Area_km2",
            "Covered_Area_km2",
            "Coverage_Percent",
        ]:
            coverage_df[col] = (
                coverage_df[col]
                .round(2)
            )

        coverage_csv = os.path.join(
            SAR_PRODUCTS_DIR,
            f"{_fp_safe_filename(platform)}_"
            f"{event['datetime'].strftime('%Y%m%d')}_"
            "District_Coverage.csv"
        )

        coverage_df.to_csv(
            coverage_csv,
            index=False
        )

        print(
            "Coverage table saved:",
            coverage_csv
        )

    # --------------------------------------------------------
    # Titles
    # --------------------------------------------------------

    event_title = (
        f"{platform} SAR"
    )

    if mode:
        event_title += (
            f"-{mode}"
        )

    # --------------------------------------------------------
    # OVERALL ASSAM MAP
    # Keep the full scene footprints visible, matching the user's
    # reference report page.
    # --------------------------------------------------------

    all_bounds = [
        assam_web.total_bounds
    ]

    all_bounds.extend(
        [
            scene["scene_web"].total_bounds
            for scene in scenes
        ]
    )

    map_xmin = min(
        b[0]
        for b in all_bounds
    )
    map_ymin = min(
        b[1]
        for b in all_bounds
    )
    map_xmax = max(
        b[2]
        for b in all_bounds
    )
    map_ymax = max(
        b[3]
        for b in all_bounds
    )

    xbuffer = (
        map_xmax - map_xmin
    ) * 0.06

    ybuffer = (
        map_ymax - map_ymin
    ) * 0.06

    fig, ax = plt.subplots(
        figsize=(17, 11)
    )

    ax.set_xlim(
        map_xmin - xbuffer,
        map_xmax + xbuffer
    )

    ax.set_ylim(
        map_ymin - ybuffer,
        map_ymax + ybuffer
    )

    _fp_add_basemap(
        ax
    )

    _fp_draw_scenes(
        ax,
        scenes,
        SAR_ALPHA_OVERALL,
        draw_footprints=True
    )

    assam_web.boundary.plot(
        ax=ax,
        edgecolor=FOOTPRINT_ASSAM_COLOR,
        linewidth=1.25,
        zorder=7,
    )

    ax.set_title(
        f"{event_title} Scene Coverage over Assam – "
        f"{event['date_text']} "
        f"({len(scenes)} Scenes)",
        fontsize=16,
        pad=12,
    )

    ax.set_axis_off()

    plt.tight_layout()

    overall_path = os.path.join(
        SAR_PRODUCTS_DIR,
        f"{_fp_safe_filename(event_title)}_"
        f"{event['datetime'].strftime('%Y%m%d')}_"
        "Overall_Assam_Coverage.png"
    )

    fig.savefig(
        overall_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        fig
    )

    print(
        "\nOverall satellite coverage map saved:",
        overall_path
    )

    # --------------------------------------------------------
    # DISTRICT PRODUCTS
    # --------------------------------------------------------

    coverage_lookup = {}
    sar_lookup = {}

    for district in target_districts:

        district_u = (
            str(district)
            .strip()
            .upper()
        )

        district_rows = assam_web[
            assam_web[DISTRICT_FIELD]
            .astype(str)
            .str.strip()
            .str.upper()
            == district_u
        ].copy()

        if district_rows.empty:

            print(
                "Satellite map warning - "
                "district not found:",
                district_u
            )

            continue

        try:
            district_geom = (
                district_rows
                .geometry
                .union_all()
            )
        except AttributeError:
            district_geom = (
                district_rows
                .geometry
                .unary_union
            )

        district_scenes = []

        for scene in scenes:

            scene_geom = (
                scene["scene_web"]
                .geometry
                .iloc[0]
            )

            if scene_geom.intersects(
                district_geom
            ):
                district_scenes.append(
                    scene
                )

        if not district_scenes:

            print(
                "No satellite scene over district:",
                district_u
            )

            continue

        row = coverage_df[
            coverage_df["District"]
            .astype(str)
            .str.upper()
            == district_u
        ]

        if row.empty:
            district_area = 0.0
            covered_area = 0.0
            coverage_pct = 0.0

        else:
            district_area = float(
                row[
                    "District_Area_km2"
                ].iloc[0]
            )

            covered_area = float(
                row[
                    "Covered_Area_km2"
                ].iloc[0]
            )

            coverage_pct = float(
                row[
                    "Coverage_Percent"
                ].iloc[0]
            )

        # District zoom.
        db = district_rows.total_bounds

        dx = max(
            db[2] - db[0],
            1.0
        )

        dy = max(
            db[3] - db[1],
            1.0
        )

        district_xlim = (
            db[0] - dx * 0.20,
            db[2] + dx * 0.20
        )

        district_ylim = (
            db[1] - dy * 0.20,
            db[3] + dy * 0.20
        )

        # ----------------------------------------------------
        # A. DISTRICT COVERAGE PAGE IMAGE
        # SAR + OSM + footprints + district boundary + coverage
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(17, 11)
        )

        ax.set_xlim(
            *district_xlim
        )

        ax.set_ylim(
            *district_ylim
        )

        _fp_add_basemap(
            ax
        )

        _fp_draw_scenes(
            ax,
            district_scenes,
            SAR_ALPHA_DISTRICT,
            draw_footprints=True
        )

        # Show neighboring district boundaries for geographic context.
        assam_web.boundary.plot(
            ax=ax,
            edgecolor=FOOTPRINT_ASSAM_COLOR,
            linewidth=0.80,
            zorder=7,
        )

        district_rows.plot(
            ax=ax,
            facecolor="none",
            edgecolor=FOOTPRINT_DISTRICT_COLOR,
            linewidth=3.0,
            zorder=9,
        )

        info_text = (
            f"{district.title()} District\n"
            f"District Area: "
            f"{district_area:,.1f} km²\n"
            f"Scene Coverage: "
            f"{covered_area:,.1f} km² "
            f"({coverage_pct:.1f}%)"
        )

        ax.text(
            0.02,
            0.97,
            info_text,
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
            ha="left",
            va="top",
            bbox=dict(
                facecolor="white",
                edgecolor=(
                    FOOTPRINT_DISTRICT_COLOR
                ),
                alpha=0.88,
                boxstyle="round,pad=0.5",
            ),
            zorder=20,
        )

        ax.set_title(
            f"Area Coverage of "
            f"{district.title()} District "
            f"from {event_title} Imageries "
            f"on {event['date_text']}",
            fontsize=15,
            pad=12,
        )

        ax.set_axis_off()

        plt.tight_layout()

        coverage_path = os.path.join(
            SAR_PRODUCTS_DIR,
            f"{_fp_safe_filename(district_u)}_"
            "Coverage.png"
        )

        fig.savefig(
            coverage_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(
            fig
        )

        coverage_lookup[
            district_u
        ] = coverage_path

        print(
            "District coverage map saved:",
            coverage_path
        )

        # ----------------------------------------------------
        # B. CLEAN DISTRICT SAR IMAGE FOR THREE-MAP PANEL
        # SAR + OSM + district boundary, but NO footprint / info box.
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(9, 7)
        )

        ax.set_xlim(
            *district_xlim
        )

        ax.set_ylim(
            *district_ylim
        )

        _fp_add_basemap(
            ax
        )

        _fp_draw_scenes(
            ax,
            district_scenes,
            0.72,
            draw_footprints=False
        )

        district_rows.plot(
            ax=ax,
            facecolor="none",
            edgecolor="red",
            linewidth=1.8,
            zorder=9,
        )

        ax.set_title(
            f"{event_title} Image of "
            f"{district.title()} District",
            fontsize=12,
            fontweight="bold"
        )

        ax.set_axis_off()

        plt.tight_layout()

        sar_path = os.path.join(
            SAR_PRODUCTS_DIR,
            f"{_fp_safe_filename(district_u)}_"
            "SAR_image.png"
        )

        fig.savefig(
            sar_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(
            fig
        )

        sar_lookup[
            district_u
        ] = sar_path

        print(
            "District SAR image saved:",
            sar_path
        )

    return (
        overall_path,
        coverage_lookup,
        sar_lookup
    )


# ============================================================
# 2. BASIC HELPERS
# ============================================================

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def norm(text):
    if pd.isna(text):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def safe_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def fmt_num(value, decimals=2):
    if pd.isna(value):
        return ""
    try:
        return f"{float(value):.{decimals}f}"
    except Exception:
        return safe_text(value)


def check_required_fields(gdf, fields, layer_name):
    missing = [f for f in fields if f not in gdf.columns]
    if missing:
        raise ValueError(
            f"{layer_name} is missing required field(s): {missing}\n"
            f"Available fields: {list(gdf.columns)}"
        )


def valid_geometries(gdf):
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    bad = ~gdf.geometry.is_valid
    if bad.any():
        warnings.warn(
            f"Repairing {bad.sum()} invalid geometries using buffer(0)."
        )
        gdf.loc[bad, "geometry"] = gdf.loc[bad, "geometry"].buffer(0)

    return gdf


def overlay_intersection(a, b):
    if a.empty or b.empty:
        return gpd.GeoDataFrame(columns=list(a.columns), geometry=[], crs=a.crs)

    return gpd.overlay(
        a,
        b,
        how="intersection",
        keep_geom_type=False
    )


def area_ha(gdf):
    if gdf.empty:
        return pd.Series(dtype=float, index=gdf.index)

    projected = gdf.to_crs(AREA_CRS)
    return projected.geometry.area / 10000.0


def dissolve_single(gdf):
    if gdf.empty:
        return gdf.copy()

    geom = gdf.geometry.unary_union
    return gpd.GeoDataFrame({"geometry": [geom]}, crs=gdf.crs)


def find_matching_file(folder, district, suffixes):
    folder = Path(folder)

    if not folder.exists():
        return None

    target = norm(district)

    for p in folder.iterdir():
        if not p.is_file():
            continue

        stem_norm = norm(p.stem)

        if target in stem_norm:
            for suffix in suffixes:
                if p.name.lower().endswith(suffix.lower()):
                    return str(p)

    allowed = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

    for p in folder.iterdir():
        if p.is_file() and p.suffix.lower() in allowed:
            if target in norm(p.stem):
                return str(p)

    return None

def read_acquisition_information(readme_pattern):
    """
    Read satellite/sensor and acquisition date/time from README.

    Expected line, for example:

    Near Real Time Inundation Mapping :
    12-Aug-2026 (RISAT MRS SAR, 1800 Hrs)

    Returns
    -------
    acquisition_date
    satellite_data
    acquisition_time
    """

    import glob
    import re

    readme_files = glob.glob(
        readme_pattern
    )

    if not readme_files:
        print(
            "WARNING: README file not found:",
            readme_pattern
        )

        return DATE, "Satellite SAR", ""

    readme_file = readme_files[0]

    print("\nREADME selected:")
    print(readme_file)

    with open(
        readme_file,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        content = f.read()

    # --------------------------------------------------------
    # Find required line
    # --------------------------------------------------------

    pattern = (
        r"Near\s+Real\s+Time\s+Inundation\s+Mapping\s*:\s*"
        r"([^(]+?)\s*"
        r"\(([^,]+),\s*([^)]+)\)"
    )

    match = re.search(
        pattern,
        content,
        flags=re.IGNORECASE
    )

    if not match:

        print(
            "WARNING: Could not extract acquisition "
            "information from README."
        )

        return DATE, "Satellite SAR", ""

    acquisition_date = (
        match.group(1).strip()
    )

    satellite_data = (
        match.group(2).strip()
    )

    acquisition_time = (
        match.group(3).strip()
    )

    print("\nAcquisition information:")
    print(
        "Date      :",
        acquisition_date
    )
    print(
        "Satellite :",
        satellite_data
    )
    print(
        "Time      :",
        acquisition_time
    )

    return (
        acquisition_date,
        satellite_data,
        acquisition_time
    )
# ============================================================
# 3. READ AND PREPARE INPUT DATA
# ============================================================

def read_input_layers():

    print("=" * 70)
    print("READING GIS LAYERS")
    print("=" * 70)

    dist_gdf = valid_geometries(gpd.read_file(DIST_SHP))
    rc_gdf = valid_geometries(gpd.read_file(RC_SHP))
    vill_gdf = valid_geometries(gpd.read_file(VILL_SHP))
    flood_gdf = valid_geometries(gpd.read_file(FLOOD_SHP))
    lulc_gdf = valid_geometries(gpd.read_file(LULC_SHP))

    check_required_fields(
        dist_gdf,
        [DISTRICT_FIELD],
        "District layer"
    )

    check_required_fields(
        rc_gdf,
        [RC_NAME_FIELD, RC_DISTRICT_FIELD, RC_AREA_FIELD],
        "Revenue-circle layer"
    )

    check_required_fields(
        vill_gdf,
        [VILLAGE_NAME_FIELD, VILLAGE_AREA_FIELD],
        "Village layer"
    )

    check_required_fields(
        lulc_gdf,
        [LULC_CLASS_FIELD],
        "LULC layer"
    )

    print("\nORIGINAL CRS")
    print("District :", dist_gdf.crs)
    print("RC       :", rc_gdf.crs)
    print("Village  :", vill_gdf.crs)
    print("Flood    :", flood_gdf.crs)
    print("LULC     :", lulc_gdf.crs)

    print("\nORIGINAL BOUNDS")
    print("District :", dist_gdf.total_bounds)
    print("Flood    :", flood_gdf.total_bounds)

    if dist_gdf.crs is None:
        raise ValueError("District shapefile does not have a CRS.")

    if flood_gdf.crs is None:
        raise ValueError("Flood shapefile does not have a CRS.")

    working_crs = dist_gdf.crs

    if rc_gdf.crs != working_crs:
        rc_gdf = rc_gdf.to_crs(working_crs)

    if vill_gdf.crs != working_crs:
        vill_gdf = vill_gdf.to_crs(working_crs)

    if flood_gdf.crs != working_crs:
        flood_gdf = flood_gdf.to_crs(working_crs)

    if lulc_gdf.crs != working_crs:
        lulc_gdf = lulc_gdf.to_crs(working_crs)

    print("\nBOUNDS AFTER CRS ALIGNMENT")
    print("District :", dist_gdf.total_bounds)
    print("Flood    :", flood_gdf.total_bounds)

    dminx, dminy, dmaxx, dmaxy = dist_gdf.total_bounds
    fminx, fminy, fmaxx, fmaxy = flood_gdf.total_bounds

    bbox_overlap = not (
        fmaxx < dminx or
        fminx > dmaxx or
        fmaxy < dminy or
        fminy > dmaxy
    )

    print("\nBounding-box overlap :", bbox_overlap)

    if not bbox_overlap:
        raise RuntimeError(
            "\nDistrict and flood layers do not overlap spatially.\n"
            f"District bounds : {dist_gdf.total_bounds}\n"
            f"Flood bounds    : {flood_gdf.total_bounds}"
        )

    dist_gdf[DISTRICT_FIELD] = (
        dist_gdf[DISTRICT_FIELD]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    rc_gdf[RC_NAME_FIELD] = (
        rc_gdf[RC_NAME_FIELD]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    rc_gdf[RC_DISTRICT_FIELD] = (
        rc_gdf[RC_DISTRICT_FIELD]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    vill_gdf[VILLAGE_NAME_FIELD] = (
        vill_gdf[VILLAGE_NAME_FIELD]
        .astype(str)
        .str.strip()
    )

    return (
        dist_gdf,
        rc_gdf,
        vill_gdf,
        flood_gdf,
        lulc_gdf
    )


# ============================================================
# 4. DETECT FLOOD-AFFECTED DISTRICTS
# ============================================================

def get_flood_districts(dist_gdf, flood_gdf):

    print("\n" + "=" * 70)
    print("DETECTING FLOOD-AFFECTED DISTRICTS")
    print("=" * 70)

    if flood_gdf.empty:
        raise RuntimeError("Flood layer contains no valid features.")

    flood_union = flood_gdf.geometry.unary_union

    affected = dist_gdf[
        dist_gdf.geometry.intersects(flood_union)
    ].copy()

    if affected.empty:
        return []

    districts = (
        affected[DISTRICT_FIELD]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
        .unique()
        .tolist()
    )

    return sorted(districts)


def filter_flood_districts_by_min_area(dist_gdf, flood_gdf, districts):
    """
    Remove districts having only a very small flood intersection.

    The area is calculated from the flood layer clipped to each district in
    AREA_CRS. This is useful for rejecting tiny edge/sliver intersections
    caused by partial satellite-scene coverage.
    """
    if MIN_DISTRICT_INUNDATED_AREA_HA <= 0:
        return list(districts)

    retained = []

    print("\nApplying minimum district inundation-area filter:")
    print(f"  Threshold : {MIN_DISTRICT_INUNDATED_AREA_HA:.2f} ha")

    for district in districts:
        district_geom = dist_gdf[
            dist_gdf[DISTRICT_FIELD] == district
        ][["geometry"]].copy()

        district_flood = overlay_intersection(
            flood_gdf[["geometry"]],
            district_geom
        )

        inundated_ha = float(area_ha(district_flood).sum())

        if inundated_ha >= MIN_DISTRICT_INUNDATED_AREA_HA:
            retained.append(district)
            print(f"  KEEP   {district:<25} {inundated_ha:10.2f} ha")
        else:
            print(f"  REJECT {district:<25} {inundated_ha:10.2f} ha")

    return retained


# ============================================================
# 5. CALCULATE DISTRICT STATISTICS
# ============================================================

def calculate_district_statistics(
    district,
    dist_gdf,
    rc_gdf,
    vill_gdf,
    flood_gdf,
    lulc_gdf
):

    district_geom = dist_gdf[
        dist_gdf[DISTRICT_FIELD] == district
    ].copy()

    district_flood = overlay_intersection(
        flood_gdf[["geometry"]],
        district_geom[["geometry"]]
    )

    rc_district = rc_gdf[
        rc_gdf[RC_DISTRICT_FIELD] == district
    ].copy()

    rc_flood_inter = overlay_intersection(
        rc_district[[RC_NAME_FIELD, "geometry"]],
        district_flood[["geometry"]]
    )

    affected_rcs = []

    if not rc_flood_inter.empty:
        affected_rcs = sorted(
            rc_flood_inter[RC_NAME_FIELD]
            .dropna()
            .astype(str)
            .str.strip()
            .str.upper()
            .unique()
            .tolist()
        )

    rc_records = []
    village_records = []

    village_area_lookup = (
        vill_gdf[
            [VILLAGE_NAME_FIELD, VILLAGE_AREA_FIELD]
        ]
        .drop_duplicates(subset=[VILLAGE_NAME_FIELD])
        .set_index(VILLAGE_NAME_FIELD)[VILLAGE_AREA_FIELD]
        .to_dict()
    )

    for rc_name in affected_rcs:

        print(f"    RC: {rc_name}")

        rc_boundary = rc_district[
            rc_district[RC_NAME_FIELD] == rc_name
        ].copy()

        if rc_boundary.empty:
            continue

        rc_boundary_one = dissolve_single(rc_boundary)

        rc_area_values = pd.to_numeric(
            rc_boundary[RC_AREA_FIELD],
            errors="coerce"
        ).dropna()

        if not rc_area_values.empty:
            rc_area_ha = float(rc_area_values.iloc[0])
        else:
            rc_area_ha = float(area_ha(rc_boundary_one).sum())

        # ----------------------------------------------------
        # VILLAGE STATISTICS
        # ----------------------------------------------------

        villages_in_rc = overlay_intersection(
            vill_gdf[
                [VILLAGE_NAME_FIELD, VILLAGE_AREA_FIELD, "geometry"]
            ],
            rc_boundary_one[["geometry"]]
        )

        if not villages_in_rc.empty:

            village_flood_inter = overlay_intersection(
                villages_in_rc[
                    [VILLAGE_NAME_FIELD, VILLAGE_AREA_FIELD, "geometry"]
                ],
                district_flood[["geometry"]]
            )

            if not village_flood_inter.empty:

                village_flood_inter["__flood_ha"] = area_ha(
                    village_flood_inter
                )

                grouped = (
                    village_flood_inter
                    .groupby(VILLAGE_NAME_FIELD, as_index=False)
                    ["__flood_ha"]
                    .sum()
                )

                for _, row in grouped.iterrows():

                    village_name = safe_text(
                        row[VILLAGE_NAME_FIELD]
                    )

                    inundated_ha = float(
                        row["__flood_ha"]
                    )

                    if inundated_ha < MIN_VILLAGE_INUNDATED_AREA_HA:
                        continue

                    village_area_value = pd.to_numeric(
                        pd.Series(
                            [village_area_lookup.get(village_name)]
                        ),
                        errors="coerce"
                    ).iloc[0]

                    if (
                        not pd.isna(village_area_value)
                        and float(village_area_value) > 0
                    ):
                        inundated_pct = (
                            inundated_ha /
                            float(village_area_value) *
                            100.0
                        )
                    else:
                        inundated_pct = np.nan

                    village_records.append({
                        "District": district,
                        "Revenue Circle": rc_name,
                        "Inundated Villages": village_name,
                        "Village Area (Ha)": village_area_value,
                        "Inundated Area (Ha)": inundated_ha,
                        "Inundated Area in %": inundated_pct
                    })

        # ----------------------------------------------------
        # LULC STATISTICS
        # ----------------------------------------------------

        lulc_in_rc = overlay_intersection(
            lulc_gdf[[LULC_CLASS_FIELD, "geometry"]],
            rc_boundary_one[["geometry"]]
        )

        lulc_flood_inter = overlay_intersection(
            lulc_in_rc[[LULC_CLASS_FIELD, "geometry"]],
            district_flood[["geometry"]]
        )

        if not lulc_flood_inter.empty:

            lulc_flood_inter["__flood_ha"] = area_ha(
                lulc_flood_inter
            )

            agri_val = float(
                lulc_flood_inter.loc[
                    lulc_flood_inter[LULC_CLASS_FIELD] ==
                    AGRICULTURE_SOURCE_NAME,
                    "__flood_ha"
                ].sum()
            )

            built_val = float(
                lulc_flood_inter.loc[
                    lulc_flood_inter[LULC_CLASS_FIELD] ==
                    BUILTUP_SOURCE_NAME,
                    "__flood_ha"
                ].sum()
            )

            other_val = float(
                lulc_flood_inter.loc[
                    ~lulc_flood_inter[LULC_CLASS_FIELD].isin(
                        [
                            AGRICULTURE_SOURCE_NAME,
                            BUILTUP_SOURCE_NAME
                        ]
                    ),
                    "__flood_ha"
                ].sum()
            )

            if agri_val > 0:
                rc_records.append({
                    "District": district,
                    "Revenue Circle (RC)": rc_name,
                    "RC Area (Ha)": rc_area_ha,
                    "LULC": "Agriculture",
                    "Inundated area (Ha)": agri_val
                })

            if built_val > 0:
                rc_records.append({
                    "District": district,
                    "Revenue Circle (RC)": rc_name,
                    "RC Area (Ha)": rc_area_ha,
                    "LULC": "Built Up",
                    "Inundated area (Ha)": built_val
                })

            if other_val > 0:
                rc_records.append({
                    "District": district,
                    "Revenue Circle (RC)": rc_name,
                    "RC Area (Ha)": rc_area_ha,
                    "LULC": "Other",
                    "Inundated area (Ha)": other_val
                })

    # --------------------------------------------------------
    # RC DATAFRAME
    # --------------------------------------------------------

    rc_df = pd.DataFrame(rc_records)

    if rc_df.empty:

        rc_df = pd.DataFrame(columns=[
            "SL.No",
            "District",
            "Revenue Circle (RC)",
            "RC Area (Ha)",
            "LULC",
            "Inundated area (Ha)"
        ])

    else:

        rc_df = rc_df[
            [
                "District",
                "Revenue Circle (RC)",
                "RC Area (Ha)",
                "LULC",
                "Inundated area (Ha)"
            ]
        ].copy()

        rc_df.insert(
            0,
            "SL.No",
            np.arange(1, len(rc_df) + 1)
        )

        rc_df["RC Area (Ha)"] = pd.to_numeric(
            rc_df["RC Area (Ha)"],
            errors="coerce"
        ).round(2)

        rc_df["Inundated area (Ha)"] = pd.to_numeric(
            rc_df["Inundated area (Ha)"],
            errors="coerce"
        ).round(2)

    # --------------------------------------------------------
    # VILLAGE DATAFRAME
    # --------------------------------------------------------

    village_df = pd.DataFrame(village_records)

    if village_df.empty:

        village_df = pd.DataFrame(columns=[
            "SL.No",
            "Revenue Circle",
            "Inundated Villages",
            "Village Area (Ha)",
            "Inundated Area (Ha)",
            "Inundated Area in %"
        ])

    else:

        village_df = (
            village_df
            .sort_values(
                ["Revenue Circle", "Inundated Villages"]
            )
            .reset_index(drop=True)
        )

        village_df.insert(
            0,
            "SL.No",
            np.arange(1, len(village_df) + 1)
        )

        for col in [
            "Village Area (Ha)",
            "Inundated Area (Ha)",
            "Inundated Area in %"
        ]:
            village_df[col] = pd.to_numeric(
                village_df[col],
                errors="coerce"
            ).round(2)

        village_df = village_df[
            [
                "SL.No",
                "Revenue Circle",
                "Inundated Villages",
                "Village Area (Ha)",
                "Inundated Area (Ha)",
                "Inundated Area in %"
            ]
        ]

    total_village_area = pd.to_numeric(
        village_df.get(
            "Village Area (Ha)",
            pd.Series(dtype=float)
        ),
        errors="coerce"
    ).sum()

    total_inundated_area = pd.to_numeric(
        village_df.get(
            "Inundated Area (Ha)",
            pd.Series(dtype=float)
        ),
        errors="coerce"
    ).sum()

    if total_village_area > 0:
        total_pct = (
            total_inundated_area /
            total_village_area *
            100.0
        )
    else:
        total_pct = np.nan

    # District-level areas are calculated directly from the district boundary
    # and the flood layer clipped to the district. This makes the first
    # summary table independent of village-boundary coverage.
    district_area_ha = float(area_ha(district_geom).sum())
    district_inundated_area_ha = float(area_ha(district_flood).sum())

    district_summary = {
        "District": district,
        "District Area (Ha)": round(district_area_ha, 2),
        "Inundated Area (Ha)": round(district_inundated_area_ha, 2),
        "No. of Revenue Circles": len(affected_rcs),
        "No. of Inundated Villages": len(village_df),
        "Village Area Total (Ha)": round(total_village_area, 2),
        "Inundated Area Total (Ha)": round(total_inundated_area, 2),
        "Overall Inundated Area (%)":
            round(total_pct, 2) if not pd.isna(total_pct) else np.nan,
        "Agriculture Inundated (Ha)": round(
            pd.to_numeric(
                rc_df.loc[
                    rc_df["LULC"] == "Agriculture",
                    "Inundated area (Ha)"
                ],
                errors="coerce"
            ).sum(),
            2
        ),
        "Built Up Inundated (Ha)": round(
            pd.to_numeric(
                rc_df.loc[
                    rc_df["LULC"] == "Built Up",
                    "Inundated area (Ha)"
                ],
                errors="coerce"
            ).sum(),
            2
        ),
        "Other Inundated (Ha)": round(
            pd.to_numeric(
                rc_df.loc[
                    rc_df["LULC"] == "Other",
                    "Inundated area (Ha)"
                ],
                errors="coerce"
            ).sum(),
            2
        )
    }

    return (
        district_geom,
        district_flood,
        rc_district,
        affected_rcs,
        rc_df,
        village_df,
        district_summary
    )


# ============================================================
# 6. CREATE DISTRICT INUNDATION MAP
# ============================================================

def create_inundation_map(
    district,
    district_geom,
    district_flood,
    rc_district,
    affected_rcs,
    vill_gdf
):

    ensure_dir(INUNDATION_MAP_DIR)

    out_png = os.path.join(
        INUNDATION_MAP_DIR,
        f"{district.title()}_inundation.png"
    )

    fig, ax = plt.subplots(figsize=(9, 8))

    villages_in_district = overlay_intersection(
        vill_gdf[["geometry"]],
        district_geom[["geometry"]]
    )

    if not villages_in_district.empty:
        villages_in_district.to_crs("EPSG:4326").plot(
            ax=ax,
            facecolor="none",
            edgecolor="slategray",
            linewidth=0.35
        )

    rc_affected = rc_district[
        rc_district[RC_NAME_FIELD].isin(affected_rcs)
    ].copy()

    if not rc_affected.empty:

        rc_ll = rc_affected.to_crs("EPSG:4326")

        rc_ll.plot(
            ax=ax,
            facecolor="none",
            edgecolor="m",
            linewidth=1.0
        )

        label_points = rc_ll.copy()
        label_points["geometry"] = (
            label_points.geometry.representative_point()
        )

        for _, row in label_points.iterrows():

            p = row.geometry

            txt = ax.annotate(
                row[RC_NAME_FIELD],
                xy=(p.x, p.y),
                ha="center",
                va="center",
                color="m",
                fontsize=9,
                fontweight="bold"
            )

            txt.set_path_effects([
                PathEffects.withStroke(
                    linewidth=3,
                    foreground="white"
                )
            ])

    district_geom.to_crs("EPSG:4326").plot(
        ax=ax,
        facecolor="none",
        edgecolor="black",
        linewidth=1.4
    )

    if not district_flood.empty:
        district_flood.to_crs("EPSG:4326").plot(
            ax=ax,
            facecolor="blue",
            edgecolor="blue",
            linewidth=0.2
        )

    ax.set_title(
        f"Inundation Map of {district.title()} District",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    legend_handles = [
        Patch(
            facecolor="none",
            edgecolor="slategray",
            label="Village boundary"
        ),
        Patch(
            facecolor="none",
            edgecolor="m",
            label="Revenue Circle boundary"
        ),
        Patch(
            facecolor="none",
            edgecolor="black",
            label="District boundary"
        ),
        Patch(
            facecolor="blue",
            edgecolor="blue",
            label="Flood inundation"
        )
    ]

    ax.legend(
        handles=legend_handles,
        loc="lower right",
        fontsize=8,
        title="Legend"
    )

    ax.set_aspect("equal", adjustable="datalim")

    fig.tight_layout()
    fig.savefig(
        out_png,
        dpi=250,
        bbox_inches="tight"
    )
    plt.close(fig)

    return out_png


# ============================================================
# 7. WORD HELPERS
# ============================================================

def set_run_font(run, size=9, bold=False, italic=False):

    run.font.name = FONT_NAME
    run.font.size = Pt(size)
    run.bold = bold
    run.italic = italic

    rpr = run._element.get_or_add_rPr()

    rfonts = rpr.rFonts

    if rfonts is None:
        rfonts = OxmlElement("w:rFonts")
        rpr.insert(0, rfonts)

    rfonts.set(qn("w:ascii"), FONT_NAME)
    rfonts.set(qn("w:hAnsi"), FONT_NAME)
    rfonts.set(qn("w:eastAsia"), FONT_NAME)


def set_cell_shading(cell, fill):

    shading = parse_xml(
        r'<w:shd {} w:fill="{}"/>'.format(
            nsdecls("w"),
            fill
        )
    )

    cell._tc.get_or_add_tcPr().append(shading)


def set_cell_margins(
    cell,
    top=45,
    start=55,
    bottom=45,
    end=55
):

    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()

    tcMar = tcPr.first_child_found_in("w:tcMar")

    if tcMar is None:
        tcMar = OxmlElement("w:tcMar")
        tcPr.append(tcMar)

    for margin, value in {
        "top": top,
        "start": start,
        "bottom": bottom,
        "end": end
    }.items():

        node = tcMar.find(qn(f"w:{margin}"))

        if node is None:
            node = OxmlElement(f"w:{margin}")
            tcMar.append(node)

        node.set(qn("w:w"), str(value))
        node.set(qn("w:type"), "dxa")


def set_fixed_table_layout(table):

    tblPr = table._tbl.tblPr

    layout = tblPr.find(qn("w:tblLayout"))

    if layout is None:
        layout = OxmlElement("w:tblLayout")
        tblPr.append(layout)

    layout.set(qn("w:type"), "fixed")



def set_table_width_cm(table, width_cm):
    """
    Force the preferred width of the whole Word table.
    This is stronger than setting cell.width alone.
    """
    tblPr = table._tbl.tblPr

    tblW = tblPr.find(qn("w:tblW"))
    if tblW is None:
        tblW = OxmlElement("w:tblW")
        tblPr.append(tblW)

    tblW.set(qn("w:w"), str(int(width_cm * 567)))
    tblW.set(qn("w:type"), "dxa")


def set_table_grid_widths(table, widths_cm):
    """
    Force Word's internal table grid widths.

    python-docx cell.width is not sufficient for nested tables;
    Word may recalculate the grid and push the RHS beyond the page.
    """
    tblGrid = table._tbl.tblGrid

    # Remove existing gridCol definitions.
    for child in list(tblGrid):
        tblGrid.remove(child)

    for width_cm in widths_cm:
        grid_col = OxmlElement("w:gridCol")
        grid_col.set(
            qn("w:w"),
            str(int(width_cm * 567))
        )
        tblGrid.append(grid_col)


def set_table_indent_zero(table):
    """
    Explicitly remove Word table indentation.

    This is especially important for nested tables because Word can otherwise
    add a small implicit offset that pushes the right-hand table toward or
    beyond the page edge.
    """

    tblPr = table._tbl.tblPr

    tblInd = tblPr.find(qn("w:tblInd"))

    if tblInd is None:
        tblInd = OxmlElement("w:tblInd")
        tblPr.append(tblInd)

    tblInd.set(qn("w:w"), "0")
    tblInd.set(qn("w:type"), "dxa")


def remove_table_borders(table):

    tblPr = table._tbl.tblPr

    borders = tblPr.find(qn("w:tblBorders"))

    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tblPr.append(borders)

    for edge in (
        "top",
        "left",
        "bottom",
        "right",
        "insideH",
        "insideV"
    ):

        tag = qn(f"w:{edge}")

        el = borders.find(tag)

        if el is None:
            el = OxmlElement(f"w:{edge}")
            borders.append(el)

        el.set(qn("w:val"), "nil")


def prevent_row_split(row):

    trPr = row._tr.get_or_add_trPr()

    cantSplit = OxmlElement("w:cantSplit")

    trPr.append(cantSplit)


def repeat_header_row(row):

    trPr = row._tr.get_or_add_trPr()

    tblHeader = OxmlElement("w:tblHeader")
    tblHeader.set(qn("w:val"), "true")

    trPr.append(tblHeader)


def set_cell_width(cell, width_cm):

    cell.width = Cm(width_cm)

    tcPr = cell._tc.get_or_add_tcPr()

    tcW = tcPr.find(qn("w:tcW"))

    if tcW is None:
        tcW = OxmlElement("w:tcW")
        tcPr.append(tcW)

    tcW.set(
        qn("w:w"),
        str(int(width_cm * 567))
    )

    tcW.set(
        qn("w:type"),
        "dxa"
    )


def format_cell(
    cell,
    font_size=8,
    bold=False,
    align=WD_ALIGN_PARAGRAPH.LEFT
):

    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    set_cell_margins(cell)

    for p in cell.paragraphs:

        p.alignment = align
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(0)
        p.paragraph_format.line_spacing = 1.0

        for run in p.runs:

            set_run_font(
                run,
                size=font_size,
                bold=bold
            )


def keep_paragraph_with_next(paragraph):

    pPr = paragraph._p.get_or_add_pPr()

    keepNext = OxmlElement("w:keepNext")

    pPr.append(keepNext)


def add_centered_heading(
    doc,
    text,
    size=13,
    space_before=0,
    space_after=6
):

    p = doc.add_paragraph()

    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    p.paragraph_format.space_before = Pt(
        space_before
    )

    p.paragraph_format.space_after = Pt(
        space_after
    )

    run = p.add_run(text)

    set_run_font(
        run,
        size=size,
        bold=True
    )

    return p


def add_picture_in_cell(
    cell,
    path,
    width_inches
):

    p = cell.paragraphs[0]

    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)

    if path and os.path.exists(path):

        try:

            run = p.add_run()

            run.add_picture(
                path,
                width=Inches(width_inches)
            )

            return

        except Exception as exc:

            print(
                f"WARNING: Could not insert image {path}: {exc}"
            )

    run = p.add_run(
        "[Image not available]"
    )

    set_run_font(
        run,
        size=9
    )


def configure_document(doc):
    """
    Landscape A4 with compact but safe margins.

    Horizontal margins are slightly larger than before so the right-hand
    nested village table never extends beyond the printable page.
    """

    section = doc.sections[0]

    section.orientation = WD_ORIENT.LANDSCAPE
    section.page_width = Cm(29.7)
    section.page_height = Cm(21.0)

    section.top_margin = Cm(0.65)
    section.bottom_margin = Cm(0.65)

    # Safer left/right margins for Word/PDF conversion.
    section.left_margin = Cm(0.90)
    section.right_margin = Cm(0.90)

    section.header_distance = Cm(0.30)
    section.footer_distance = Cm(0.30)

    style = doc.styles["Normal"]
    style.font.name = FONT_NAME
    style.font.size = Pt(9)


def add_page_number(section):

    footer = section.footer

    p = footer.paragraphs[0]

    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    run = p.add_run()

    fldChar1 = OxmlElement("w:fldChar")
    fldChar1.set(qn("w:fldCharType"), "begin")

    instrText = OxmlElement("w:instrText")
    instrText.set(qn("xml:space"), "preserve")
    instrText.text = " PAGE "

    fldChar2 = OxmlElement("w:fldChar")
    fldChar2.set(qn("w:fldCharType"), "end")

    run._r.append(fldChar1)
    run._r.append(instrText)
    run._r.append(fldChar2)


# ============================================================
# 8. MAP PANEL
# ============================================================

def standardize_map_image(
    input_path,
    output_path,
    target_size=(1600, 1080),
    white_threshold=250,
    padding=8
):
    """
    Standardize map images without clipping titles, legends, north arrows,
    coordinate labels or scale bars.

    Only the outer near-white border is removed. The remaining complete map
    panel is then resized to the requested target dimensions.

    Satellite and LULC are therefore made exactly the same raster size before
    insertion into Word.
    """

    if not input_path or not os.path.exists(input_path):
        return None

    try:
        img = Image.open(input_path).convert("RGB")
        arr = np.asarray(img)

        # Detect meaningful content including map frame, text and legend.
        non_white = np.any(arr < white_threshold, axis=2)

        rows = np.where(np.any(non_white, axis=1))[0]
        cols = np.where(np.any(non_white, axis=0))[0]

        if len(rows) > 0 and len(cols) > 0:
            top = max(0, int(rows[0]) - padding)
            bottom = min(img.height, int(rows[-1]) + padding + 1)
            left = max(0, int(cols[0]) - padding)
            right = min(img.width, int(cols[-1]) + padding + 1)

            img = img.crop((left, top, right, bottom))

        # Resize the complete remaining map panel.
        img = img.resize(
            target_size,
            resample=Image.Resampling.LANCZOS
        )

        img.save(
            output_path,
            quality=96,
            subsampling=0
        )

        return output_path

    except Exception as exc:
        print(
            f"WARNING: Could not standardize {input_path}: {exc}"
        )
        return input_path


def add_picture_exact(cell, image_path, width_inches, height_inches):
    """
    Insert an already-standardized image at an exact Word display size.
    """
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)

    if image_path and os.path.exists(image_path):
        try:
            run = p.add_run()
            run.add_picture(
                image_path,
                width=Inches(width_inches),
                height=Inches(height_inches)
            )
            return
        except Exception as exc:
            print(f"WARNING: Could not insert image {image_path}: {exc}")

    run = p.add_run("[Image not available]")
    set_run_font(run, size=9)


def add_district_image_panel(
    doc,
    district,
    satellite_img,
    flood_img,
    lulc_img
):
    """
    Follow the requested reference pattern:

        Satellite image        |  Flood inundation map
        -----------------------|  spanning both rows
        LULC map               |

    The satellite and LULC maps have identical display dimensions.
    The inundation map is taller and occupies the full right side.
    """

    # Prepare standardized canvases. This avoids distortion while allowing
    # exact, balanced Word dimensions.
    map_cache = Path(OUTPUT_ROOT) / "report_map_images"
    map_cache.mkdir(parents=True, exist_ok=True)

    safe_district = re.sub(r"[^A-Za-z0-9_-]+", "_", district.title())

    # Satellite and LULC MUST be created with exactly the same raster size.
    LEFT_TARGET_SIZE = (1600, 1080)

    sat_std = standardize_map_image(
        satellite_img,
        str(map_cache / f"{safe_district}_satellite.jpg"),
        LEFT_TARGET_SIZE
    )

    lulc_std = standardize_map_image(
        lulc_img,
        str(map_cache / f"{safe_district}_lulc.jpg"),
        LEFT_TARGET_SIZE
    )

    # Flood map is intentionally taller because it spans both rows on the right.
    flood_std = standardize_map_image(
        flood_img,
        str(map_cache / f"{safe_district}_inundation.jpg"),
        (1500, 1720)
    )

    # No extra global heading here: the three source maps already contain
    # their own titles, matching the reference page.
    table = doc.add_table(rows=2, cols=2)
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    set_fixed_table_layout(table)
    set_table_indent_zero(table)
    remove_table_borders(table)

    # Use almost the complete printable width.
    LEFT_WIDTH = 12.45
    RIGHT_WIDTH = 15.55

    set_cell_width(table.cell(0, 0), LEFT_WIDTH)
    set_cell_width(table.cell(1, 0), LEFT_WIDTH)
    set_cell_width(table.cell(0, 1), RIGHT_WIDTH)
    set_cell_width(table.cell(1, 1), RIGHT_WIDTH)

    satellite_cell = table.cell(0, 0)
    lulc_cell = table.cell(1, 0)
    flood_cell = table.cell(0, 1).merge(table.cell(1, 1))

    # Very small internal margins: reduce the large central/outer gaps.
    for cell in (satellite_cell, lulc_cell, flood_cell):
        set_cell_margins(
            cell,
            top=10,
            start=12,
            bottom=10,
            end=12
        )
        cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

    # Satellite and LULC use EXACTLY the same Word display box.
    LEFT_MAP_WIDTH_IN = 4.72
    LEFT_MAP_HEIGHT_IN = 3.18

    add_picture_exact(
        satellite_cell,
        sat_std,
        width_inches=LEFT_MAP_WIDTH_IN,
        height_inches=LEFT_MAP_HEIGHT_IN
    )

    add_picture_exact(
        lulc_cell,
        lulc_std,
        width_inches=LEFT_MAP_WIDTH_IN,
        height_inches=LEFT_MAP_HEIGHT_IN
    )

    add_picture_exact(
        flood_cell,
        flood_std,
        width_inches=5.80,
        height_inches=6.55
    )

    # Critical: the complete map panel stays on one page.
    for row in table.rows:
        prevent_row_split(row)



# ============================================================
# 9. GENERIC NORMAL TABLE
# ============================================================

def merge_contiguous_table_cells(table, df, column_name, group_columns=None,
                                 align=WD_ALIGN_PARAGRAPH.CENTER,
                                 font_size=8):
    """
    Vertically merge repeated values in a Word table for contiguous groups.

    ``table`` contains one header row, so DataFrame row 0 corresponds to
    Word table row 1. ``group_columns`` can be used to ensure that cells are
    merged only inside the same parent group (for example Revenue Circle
    within District).
    """
    if df.empty or column_name not in df.columns:
        return

    if group_columns is None:
        group_columns = []

    work = df.reset_index(drop=True).copy()

    def key_for(i):
        parent = tuple(safe_text(work.loc[i, col]) for col in group_columns)
        value = safe_text(work.loc[i, column_name])
        return parent + (value,)

    start = 0
    while start < len(work):
        current_key = key_for(start)
        end = start

        while end + 1 < len(work) and key_for(end + 1) == current_key:
            end += 1

        col_idx = list(work.columns).index(column_name)
        first_cell = table.cell(start + 1, col_idx)

        if end > start:
            last_cell = table.cell(end + 1, col_idx)
            merged_cell = first_cell.merge(last_cell)
            merged_cell.text = safe_text(work.loc[start, column_name])
        else:
            merged_cell = first_cell

        format_cell(
            merged_cell,
            font_size=font_size,
            bold=False,
            align=align
        )

        start = end + 1


def add_dataframe_table(
    doc,
    df,
    title,
    column_widths_cm,
    font_size=8
):

    heading = add_centered_heading(
        doc,
        title,
        size=11,
        space_before=7,
        space_after=4
    )

    keep_paragraph_with_next(
        heading
    )

    headers = list(
        df.columns
    )

    table = doc.add_table(
        rows=1,
        cols=len(headers)
    )

    table.style = "Table Grid"

    table.alignment = (
        WD_TABLE_ALIGNMENT.CENTER
    )

    table.autofit = False

    set_fixed_table_layout(
        table
    )

    set_table_indent_zero(
        table
    )

    set_table_width_cm(
        table,
        sum(column_widths_cm)
    )

    set_table_grid_widths(
        table,
        column_widths_cm
    )

    header_row = table.rows[0]

    for i, header in enumerate(headers):

        cell = header_row.cells[i]

        cell.text = str(header)

        set_cell_width(
            cell,
            column_widths_cm[i]
        )

        set_cell_shading(
            cell,
            TABLE_HEADER_FILL
        )

        format_cell(
            cell,
            font_size=font_size,
            bold=True,
            align=WD_ALIGN_PARAGRAPH.CENTER
        )

    repeat_header_row(
        header_row
    )

    prevent_row_split(
        header_row
    )

    for _, record in df.iterrows():

        row = table.add_row()

        prevent_row_split(
            row
        )
        for i, column in enumerate(headers):

            cell = row.cells[i]

    # ========================================================
    # AREA VALUES - 2 DECIMALS + RIGHT ALIGNMENT
    # ========================================================
            if column in [
                "RC Area (Ha)",
                "Inundated area (Ha)",
                "District Area (Ha)",
                "Inundated Area (Ha)"
            ]:

                cell.text = fmt_num(
                    record[column],
                    2
                )

                alignment = (
                    WD_ALIGN_PARAGRAPH.RIGHT
                )

    # ========================================================
    # COUNT VALUES - INTEGER + RIGHT ALIGNMENT
    # ========================================================
            elif column in [
                "No. of Revenue circle",
                "No. of Village"
            ]:

                if pd.isna(record[column]):
                    cell.text = ""
                else:
                    cell.text = str(
                        int(record[column])
                    )

                alignment = (
                    WD_ALIGN_PARAGRAPH.RIGHT
                )

    # ========================================================
    # SERIAL NUMBER - INTEGER + RIGHT ALIGNMENT
    # ========================================================
            elif column == "SL.No":

                cell.text = str(
                    int(record[column])
                )

                alignment = (
                    WD_ALIGN_PARAGRAPH.RIGHT
                )

    # ========================================================
    # TEXT - LEFT ALIGNMENT
    # ========================================================
            else:

                cell.text = safe_text(
                    record[column]
                )   

                alignment = (
                    WD_ALIGN_PARAGRAPH.LEFT
                )

    # ========================================================
    # APPLY WIDTH AND FORMATTING
    # ========================================================
            set_cell_width(
                cell,
                column_widths_cm[i]
            )

            format_cell(
                cell,
                font_size=font_size,
                bold=False,
                align=alignment
            )

    # RC/LULC statistics table: merge repeated District, Revenue Circle and
    # RC Area cells vertically. This produces a cleaner hierarchical table,
    # while SL.No, LULC and inundated-area values remain row-wise.
    if "Revenue Circle (RC)" in headers:
        if "District" in headers:
            merge_contiguous_table_cells(
                table, df, "District",
                group_columns=[],
                align=WD_ALIGN_PARAGRAPH.CENTER,
                font_size=font_size
            )

        merge_contiguous_table_cells(
            table, df, "Revenue Circle (RC)",
            group_columns=["District"] if "District" in headers else [],
            align=WD_ALIGN_PARAGRAPH.CENTER,
            font_size=font_size
        )

        if "RC Area (Ha)" in headers:
            merge_contiguous_table_cells(
                table, df, "RC Area (Ha)",
                group_columns=[
                    c for c in ["District", "Revenue Circle (RC)"]
                    if c in headers
                ],
                align=WD_ALIGN_PARAGRAPH.RIGHT,
                font_size=font_size
            )

    return table


# ============================================================
# 10. TWO-COLUMN VILLAGE TABLE HELPERS
# ============================================================

def set_cell_text_direction_vertical(cell):

    tcPr = cell._tc.get_or_add_tcPr()

    text_direction = tcPr.find(
        qn("w:textDirection")
    )

    if text_direction is None:

        text_direction = OxmlElement(
            "w:textDirection"
        )

        tcPr.append(
            text_direction
        )

    text_direction.set(
        qn("w:val"),
        "btLr"
    )


def estimated_row_units(village_name):
    """
    Estimate physical row height from likely text wrapping.
    Wider village columns allow larger fonts, so thresholds are generous.
    """
    text = safe_text(village_name)
    n = len(text)

    if n <= 25:
        return 1.0
    elif n <= 43:
        return 1.35
    elif n <= 62:
        return 1.75
    else:
        return 2.15


def split_village_dataframe_by_rc(village_df, max_units=32.0):
    """
    Split data into page half-columns using estimated physical height.
    Revenue-circle groups are kept intact whenever they fit.
    Large RCs continue naturally in the next half-column/page.
    """
    if village_df.empty:
        return []

    data = village_df.copy()
    data["__units"] = data["Inundated Villages"].apply(estimated_row_units)

    columns = []
    current_parts = []
    current_units = 0.0

    for rc_name, rc_group in data.groupby("Revenue Circle", sort=False):
        rc_group = rc_group.copy().reset_index(drop=True)
        group_units = float(rc_group["__units"].sum())

        # Keep a complete RC in the current column if possible.
        if current_parts and current_units + group_units <= max_units:
            current_parts.append(rc_group)
            current_units += group_units
            continue

        # If the RC fits in a fresh column, close the old one first.
        if group_units <= max_units:
            if current_parts:
                finished = pd.concat(current_parts, ignore_index=True)
                columns.append(finished.drop(columns="__units"))
            current_parts = [rc_group]
            current_units = group_units
            continue

        # The RC itself is too large: close any previous column, then split
        # this RC by estimated height. The RC label is repeated in each part.
        if current_parts:
            finished = pd.concat(current_parts, ignore_index=True)
            columns.append(finished.drop(columns="__units"))
            current_parts = []
            current_units = 0.0

        part_rows = []
        part_units = 0.0

        for _, record in rc_group.iterrows():
            units = float(record["__units"])

            if part_rows and part_units + units > max_units:
                part_df = pd.DataFrame(part_rows)
                columns.append(part_df.drop(columns="__units"))
                part_rows = []
                part_units = 0.0

            part_rows.append(record)
            part_units += units

        if part_rows:
            # Keep the final part open so the next small RC may use remaining space.
            current_parts = [pd.DataFrame(part_rows)]
            current_units = part_units

    if current_parts:
        finished = pd.concat(current_parts, ignore_index=True)
        columns.append(finished.drop(columns="__units"))

    return columns


def create_half_width_village_table(
    container_cell,
    village_data
):

    headers = [
        "SL.\nNo",
        "Revenue\nCircle",
        "Inundated Villages",
        "Village\nArea\n(Ha)",
        "Inundated\nArea (Ha)",
        "Inundated\nArea in\n%"
    ]

    table = container_cell.add_table(
        rows=1,
        cols=6
    )

    table.style = "Table Grid"
    table.autofit = False

    set_fixed_table_layout(
        table
    )

    # Remove any implicit Word indent from this nested table.
    set_table_indent_zero(
        table
    )

    # Wider half-page table with more space for village names.
    # This reduces wrapping and lets us use a larger font.
    # Conservative fixed width for a half-page nested table.
    # Total = 12.10 cm.
    # This keeps the final percentage column safely inside the page.
    widths = [
        0.65,   # SL No
        1.30,   # Revenue Circle
        4.25,   # Inundated Villages
        1.90,   # Village Area
        2.00,   # Inundated Area
        2.00    # Percentage
    ]

    set_table_width_cm(
        table,
        sum(widths)
    )

    set_table_grid_widths(
        table,
        widths
    )

    header_row = table.rows[0]

    for col_index, header in enumerate(
        headers
    ):

        cell = header_row.cells[
            col_index
        ]

        cell.text = header

        set_cell_width(
            cell,
            widths[col_index]
        )

        set_cell_shading(
            cell,
            TABLE_HEADER_FILL
        )

        format_cell(
            cell,
            font_size=VILLAGE_HEADER_FONT_SIZE,
            bold=True,
            align=WD_ALIGN_PARAGRAPH.CENTER
        )

    repeat_header_row(
        header_row
    )

    prevent_row_split(
        header_row
    )

    for _, record in village_data.iterrows():

        row = table.add_row()

        prevent_row_split(
            row
        )

        values = [
            str(
                int(
                    record["SL.No"]
                )
            ),
            safe_text(
                record["Revenue Circle"]
            ),
            safe_text(
                record["Inundated Villages"]
            ),
            fmt_num(
                record["Village Area (Ha)"],
                2
            ),
            fmt_num(
                record["Inundated Area (Ha)"],
                2
            ),
            fmt_num(
                record["Inundated Area in %"],
                2
            )
        ]

        for col_index, value in enumerate(
            values
        ):

            cell = row.cells[
                col_index
            ]

            cell.text = value

            set_cell_width(
                cell,
                widths[col_index]
            )

            if col_index in [
                0,
                3,
                4,
                5
            ]:

                alignment = (
                    WD_ALIGN_PARAGRAPH.RIGHT
                )

            elif col_index == 1:

                alignment = (
                    WD_ALIGN_PARAGRAPH.CENTER
                )

            else:

                alignment = (
                    WD_ALIGN_PARAGRAPH.LEFT
                )

            format_cell(
                cell,
                font_size=VILLAGE_FONT_SIZE,
                bold=False,
                align=alignment
            )

    # --------------------------------------------------------
    # Merge Revenue Circle cells vertically
    # --------------------------------------------------------

    if len(village_data) > 0:

        rc_values = (
            village_data[
                "Revenue Circle"
            ]
            .astype(str)
            .tolist()
        )

        start = 0

        while start < len(
            rc_values
        ):

            current_rc = (
                rc_values[start]
            )

            end = start

            while (
                end + 1 < len(rc_values)
                and
                rc_values[end + 1]
                == current_rc
            ):

                end += 1

            first_row = start + 1
            last_row = end + 1

            if last_row > first_row:

                merged_cell = table.cell(
                    first_row,
                    1
                ).merge(
                    table.cell(
                        last_row,
                        1
                    )
                )

            else:

                merged_cell = table.cell(
                    first_row,
                    1
                )

            merged_cell.text = (
                current_rc
            )

            set_cell_text_direction_vertical(
                merged_cell
            )

            merged_cell.vertical_alignment = (
                WD_CELL_VERTICAL_ALIGNMENT.CENTER
            )

            format_cell(
                merged_cell,
                font_size=RC_FONT_SIZE,
                bold=True,
                align=WD_ALIGN_PARAGRAPH.CENTER
            )

            start = end + 1

    return table


def append_total_row_to_half_table(table, village_df):
    """
    Append the grand total directly to the final half-table so it cannot
    become a detached table on the following page.
    """
    total_village_area = pd.to_numeric(
        village_df["Village Area (Ha)"], errors="coerce"
    ).sum()

    total_inundated_area = pd.to_numeric(
        village_df["Inundated Area (Ha)"], errors="coerce"
    ).sum()

    overall_pct = (
        total_inundated_area / total_village_area * 100.0
        if total_village_area > 0 else np.nan
    )

    row = table.add_row()
    prevent_row_split(row)

    # Merge first three cells for a compact "Total" label.
    label_cell = row.cells[0].merge(row.cells[2])
    label_cell.text = "Total"
    format_cell(
        label_cell,
        font_size=VILLAGE_FONT_SIZE,
        bold=True,
        align=WD_ALIGN_PARAGRAPH.RIGHT
    )

    numeric_values = [
        fmt_num(total_village_area, 2),
        fmt_num(total_inundated_area, 2),
        fmt_num(overall_pct, 2)
    ]

    for idx, value in zip((3, 4, 5), numeric_values):
        cell = row.cells[idx]
        cell.text = value
        format_cell(
            cell,
            font_size=VILLAGE_FONT_SIZE,
            bold=True,
            align=WD_ALIGN_PARAGRAPH.RIGHT
        )


def add_two_column_village_tables(
    doc,
    village_df,
    max_units=VILLAGE_MAX_UNITS
):
    """
    Two compact side-by-side village tables per page.

    Improvements:
    - uses nearly the full landscape page width;
    - only a very small central gap;
    - larger 8-pt text;
    - wider village-name column;
    - physical-height-aware pagination;
    - outer pair cannot split across pages;
    - grand total is appended to the final half-table.
    """

    heading = add_centered_heading(
        doc,
        "List of villages under flood inundation",
        size=12,
        space_before=6,
        space_after=3
    )
    keep_paragraph_with_next(heading)

    column_dfs = split_village_dataframe_by_rc(
        village_df,
        max_units=max_units
    )

    if not column_dfs:
        p = doc.add_paragraph("No inundated villages identified.")
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        return

    for column_index in range(0, len(column_dfs), 2):

        if column_index > 0:
            doc.add_page_break()
            heading = add_centered_heading(
                doc,
                "List of villages under flood inundation (Continued)",
                size=12,
                space_before=0,
                space_after=3
            )
            keep_paragraph_with_next(heading)

        outer = doc.add_table(rows=1, cols=2)
        outer.alignment = WD_TABLE_ALIGNMENT.CENTER
        outer.autofit = False
        set_fixed_table_layout(outer)

        # Explicitly remove Word's table indentation.
        set_table_indent_zero(outer)

        remove_table_borders(outer)

        # Essential: Word must not split the left/right pair between pages.
        prevent_row_split(outer.rows[0])

        left_cell = outer.cell(0, 0)
        right_cell = outer.cell(0, 1)

        # Safer two-column width.
        # A4 landscape width = 29.7 cm.
        # With 0.90 cm margins, printable width = 27.90 cm.
        # Two 13.20 cm containers use 26.40 cm, leaving 1.50 cm safety.
        HALF_TABLE_CONTAINER_WIDTH = 13.20

        set_table_width_cm(
            outer,
            HALF_TABLE_CONTAINER_WIDTH * 2
        )

        set_table_grid_widths(
            outer,
            [
                HALF_TABLE_CONTAINER_WIDTH,
                HALF_TABLE_CONTAINER_WIDTH
            ]
        )

        set_cell_width(
            left_cell,
            HALF_TABLE_CONTAINER_WIDTH
        )

        set_cell_width(
            right_cell,
            HALF_TABLE_CONTAINER_WIDTH
        )

        # Minimal centre gap.
        set_cell_margins(
            left_cell,
            top=0,
            start=0,
            bottom=0,
            end=8
        )

        set_cell_margins(
            right_cell,
            top=0,
            start=8,
            bottom=0,
            end=0
        )

        left_table = create_half_width_village_table(
            left_cell,
            column_dfs[column_index]
        )

        right_table = None

        if column_index + 1 < len(column_dfs):
            right_table = create_half_width_village_table(
                right_cell,
                column_dfs[column_index + 1]
            )

        # The total belongs to the final half-table, never as a detached table.
        is_final_pair = column_index + 2 >= len(column_dfs)

        if is_final_pair:
            if right_table is not None:
                append_total_row_to_half_table(right_table, village_df)
            else:
                append_total_row_to_half_table(left_table, village_df)



# ============================================================
# 11. ADD ONE DISTRICT TO REPORT
# ============================================================

def add_district_to_report(
    doc,
    district,
    rc_df,
    village_df,
    satellite_img,
    flood_img,
    lulc_img
):

    add_district_image_panel(
        doc,
        district,
        satellite_img,
        flood_img,
        lulc_img
    )

    # Keep the map composition as a dedicated page.
    doc.add_page_break()

    rc_widths = [
        1.2,
        3.5,
        5.0,
        2.5,
        3.0,
        3.3
    ]

    add_dataframe_table(
        doc,
        rc_df,
        "Built-up and Agricultural area under flood inundation",
        rc_widths,
        font_size=8
    )

    add_two_column_village_tables(
        doc,
        village_df,
        max_units=VILLAGE_MAX_UNITS
    )


# ============================================================
# 12. EXECUTIVE SUMMARY
# ============================================================

def add_executive_summary(
    doc,
    district_summary_df,
    acquisition_date,
    satellite_data,
    acquisition_time
):
    add_centered_heading(
        doc,
        "Executive Summary",
        size=18,
        space_after=10
    )

    no_dist = len(
        district_summary_df
    )

    no_vill = int(
        pd.to_numeric(
            district_summary_df[
                "No. of Inundated Villages"
            ],
            errors="coerce"
        ).sum()
    )

    inundated_area = pd.to_numeric(
        district_summary_df[
            "Inundated Area (Ha)"
        ],
        errors="coerce"
    ).sum()

    agriculture_area = pd.to_numeric(
        district_summary_df[
            "Agriculture Inundated (Ha)"
        ],
        errors="coerce"
    ).sum()

    builtup_area = pd.to_numeric(
        district_summary_df[
            "Built Up Inundated (Ha)"
        ],
        errors="coerce"
    ).sum()

    p = doc.add_paragraph()

    p.alignment = (
        WD_ALIGN_PARAGRAPH.JUSTIFY
    )

    p.paragraph_format.space_after = Pt(6)

    # Bring the Executive Summary content slightly inward from both sides
    # without changing the margins used by the district pages.
    p.paragraph_format.left_indent = Cm(0.80)
    p.paragraph_format.right_indent = Cm(0.80)

    text = (
        f"National Remote Sensing Centre (NRSC), Hyderabad is conducting "
        f"near real-time flood mapping for Assam using satellite data. "
        f"Flooding was reported in Assam during {acquisition_date} due to "
        f"incessant heavy rainfall. NRSC has analysed {satellite_data} "
        f"satellite data acquired on {acquisition_date}"
    )

    if acquisition_time:
        text += f" at {acquisition_time}"

    text += (
        f" and prepared a flood inundation map for parts of Assam. "
        f"NESAC has generated value-added information using the flood "
        f"inundation layers shared by NRSC. The flood inundation layer "
        f"was brought into a GIS environment to generate detailed "
        f"district-wise flood inundation information at the village level. "
        f"The details are presented at district, revenue circle and village "
        f"levels, and the latest available land use/land cover information "
        f"has been used for generation of inundation statistics. "
        f"On the date of satellite data acquisition, {no_vill:,} villages "
        f"were under flood inundation in {no_dist} districts of Assam, "
        f"covering an inundated area of {inundated_area:.2f} ha. "
        f"Of this, approximately {agriculture_area:.2f} ha comprised agricultural land. "
        f"The detailed district-wise flood inundation information is "
        f"presented below."
    )
    run = p.add_run(
        text
    )

    set_run_font(
        run,
        size=12
    )

    # First table: compact district-wise flood statistics requested for the
    # executive summary. Values remain dynamically calculated from GIS data.
    summary_display = (
        district_summary_df[
            [
                "District",
                "District Area (Ha)",
                "Inundated Area (Ha)",
                "No. of Revenue Circles",
                "No. of Inundated Villages"
            ]
        ].copy()
    )

    # User-facing column labels.
    summary_display = summary_display.rename(columns={
        "No. of Revenue Circles": "No. of Revenue circle",
        "No. of Inundated Villages": "No. of Village"
    })

    # Slightly narrower than the normal report tables so the first-page
    # summary is visually indented from the left and right edges.
    widths = [
        4.60,
        4.10,
        4.10,
        4.30,
        3.80
    ]

    add_dataframe_table(
        doc,
        summary_display,
        "Summary of water inundation based on Satellite data coverage ",
        widths,
        font_size=12
    )

    # ============================================================
    # DISCLAIMER - CENTRED NEAR BOTTOM OF FIRST PAGE
    # ============================================================

    disclaimer = doc.add_paragraph()

    # Push the disclaimer downward toward the bottom of the page
    disclaimer.paragraph_format.space_before = Pt(105)
    disclaimer.paragraph_format.space_after = Pt(4)

    # Keep some distance from the left/right page edges
    disclaimer.paragraph_format.left_indent = Cm(1.20)
    disclaimer.paragraph_format.right_indent = Cm(1.20)

    # Centre align
    disclaimer.alignment = WD_ALIGN_PARAGRAPH.CENTER

    run = disclaimer.add_run(
        "Disclaimer-The flood statistics are generated based on satellite "
        "data coverage for the area on the date. Some districts are not fully "
        "covered by the satellite."
    )

    set_run_font(
        run,
        size=10,
        italic=True
    )


# ============================================================
# SATELLITE COVERAGE PAGES FOR REPORT
# ============================================================

def add_report_image_page(
    doc,
    image_path,
    caption=None,
    max_width_inches=9.0,
    max_height_inches=6.25
):
    """
    Add one centered coverage image and keep the image + caption
    comfortably on the SAME landscape A4 page.

    The image is scaled using BOTH available width and height.
    This avoids a tall SAR coverage image consuming the whole page
    and pushing its caption to the following page.
    """

    if not image_path or not os.path.exists(image_path):

        p = doc.add_paragraph(
            "[Satellite coverage image not available]"
        )

        p.alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
        )

        return

    # --------------------------------------------------------
    # READ IMAGE ASPECT RATIO
    # --------------------------------------------------------

    try:
        with Image.open(image_path) as img:
            pixel_width, pixel_height = img.size
    except Exception:
        pixel_width, pixel_height = 1600, 1000

    if pixel_width <= 0 or pixel_height <= 0:
        pixel_width, pixel_height = 1600, 1000

    aspect = (
        float(pixel_width)
        / float(pixel_height)
    )

    # Size if constrained by width.
    draw_width = float(
        max_width_inches
    )

    draw_height = (
        draw_width
        / aspect
    )

    # If too tall, constrain by height instead.
    if draw_height > max_height_inches:

        draw_height = float(
            max_height_inches
        )

        draw_width = (
            draw_height
            * aspect
        )

    # --------------------------------------------------------
    # IMAGE PARAGRAPH
    # --------------------------------------------------------
    spacer = doc.add_paragraph()
    spacer.paragraph_format.space_before = Pt(0)
    spacer.paragraph_format.space_after = Pt(0)
    spacer.paragraph_format.line_spacing = Pt(50)
    
    spacer.add_run(" ")
    
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(3)
   # p = doc.add_paragraph()

  #  p.alignment = (
  #      WD_ALIGN_PARAGRAPH.CENTER
  #  )

  #  p.paragraph_format.space_before = Pt(50)
   # p.paragraph_format.space_after = Pt(3)

    # Keep the image paragraph with the following caption.
    if caption:
        keep_paragraph_with_next(p)

    run = p.add_run()

    run.add_picture(
        image_path,
        width=Inches(draw_width),
        height=Inches(draw_height)
    )

    # --------------------------------------------------------
    # CAPTION
    # --------------------------------------------------------

    if caption:

        cap = doc.add_paragraph()

        cap.alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
        )

        cap.paragraph_format.space_before = Pt(1)
        cap.paragraph_format.space_after = Pt(0)

        cap_run = cap.add_run(
            caption
        )

        set_run_font(
            cap_run,
            size=12,
            bold=False
        )


# ============================================================
# 13. PAGE-SPACING / END-MARKER HELPERS
# ============================================================

def add_blank_pages(doc, count=1):
    """
    Insert exactly ``count`` completely blank pages and leave the cursor
    on the page where the next content should start.

    Example:
        current content page -> blank page -> next content page
    requires two page breaks.
    """
    if count <= 0:
        return

    # Leave the current content page and enter the first blank page.
    doc.add_page_break()

    # Each blank page is then closed with another page break.
    for _ in range(count):
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(0)
        doc.add_page_break()


def add_end_marker(doc):
    """Add a simple centred end-of-report marker on the final page."""
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(8)
    p.paragraph_format.space_after = Pt(0)

    run = p.add_run("---------- **** ----------")
    set_run_font(run, size=15, bold=True)


# ============================================================
# 14. MAIN REPORT
# ============================================================

def generate_report():

    ensure_dir(
        OUTPUT_ROOT
    )

    ensure_dir(
        INUNDATION_MAP_DIR
    )

    (
        dist_gdf,
        rc_gdf,
        vill_gdf,
        flood_gdf,
        lulc_gdf
    ) = read_input_layers()

    flood_districts = (
        get_flood_districts(
            dist_gdf,
            flood_gdf
        )
    )

    flood_districts = (
        filter_flood_districts_by_min_area(
            dist_gdf,
            flood_gdf,
            flood_districts
        )
    )

    if not flood_districts:

        raise RuntimeError(
            "No district remains after applying "
            "the inundation-area filter."
        )

    print(
        "\nAffected districts:"
    )

    for district in flood_districts:
        print(
            "  -",
            district
        )

    # ========================================================
    # SATELLITE SEARCH + REAL SAR + FOOTPRINT MAPS
    # ========================================================

    overall_coverage_img = None
    district_coverage_lookup = {}
    district_sar_lookup = {}

    if AUTO_GENERATE_SATELLITE_COVERAGE:

        try:

            (
                overall_coverage_img,
                district_coverage_lookup,
                district_sar_lookup
            ) = generate_satellite_coverage_maps(
                dist_gdf,
                flood_districts
            )

        except Exception as exc:

            print(
                "\nWARNING: Automatic satellite "
                "map generation failed."
            )

            print(
                "Reason:",
                exc
            )

            if FOOTPRINT_REQUIRED:
                raise

    # ========================================================
    # CREATE REPORT + CALCULATE DISTRICT PRODUCTS
    # ========================================================

    # ========================================================
    # TITLE PAGES - ADDED WITHOUT CHANGING REPORT LOGIC
    # ========================================================

    serial_value = REPORT_SERIAL_INPUT

    if serial_value is None:
        serial_value = input(
            "\nEnter report serial number "
            "(example: 28, 0028, or complete NESAC code): "
        ).strip()

    customized_title = os.path.join(
        OUTPUT_ROOT,
        f"Title_{EVENT_DATE.strftime('%Y%m%d')}.docx"
    )

    prepare_title_template(
        TITLE_TEMPLATE,
        customized_title,
        EVENT_DATE,
        serial_value
    )

    # Start from the existing two-page title template.
    doc = Document(customized_title)

    # The actual flood report begins on a new landscape section.
    report_section = doc.add_section(WD_SECTION.NEW_PAGE)
    configure_generated_report_section(report_section)
    restart_report_page_numbering(report_section, start=1)
    add_page_number(report_section)

    # Preserve the same Normal style used by the original report code.
    style = doc.styles["Normal"]
    style.font.name = FONT_NAME
    style.font.size = Pt(9)

    all_rc = []
    all_village = []
    all_summary = []
    district_products = []

    for idx, district in enumerate(
        flood_districts,
        start=1
    ):

        print(
            f"\n[{idx}/{len(flood_districts)}] "
            f"Processing {district}"
        )

        (
            district_geom,
            district_flood,
            rc_district,
            affected_rcs,
            rc_df,
            village_df,
            summary
        ) = calculate_district_statistics(
            district,
            dist_gdf,
            rc_gdf,
            vill_gdf,
            flood_gdf,
            lulc_gdf
        )

        flood_map = create_inundation_map(
            district,
            district_geom,
            district_flood,
            rc_district,
            affected_rcs,
            vill_gdf
        )

        district_key = (
            str(district)
            .strip()
            .upper()
        )

        # ----------------------------------------------------
        # PRE-FLOOD SATELLITE IMAGE FOR ORIGINAL REPORT PANEL
        # ----------------------------------------------------
        # IMPORTANT:
        # Do NOT use the current-event SAR image here.
        #
        # The original district report panel must remain:
        #
        #   Pre-flood satellite image | Inundation map
        #   LULC map                  | Inundation map
        #
        # The current-event SAR imagery is used only on the separate
        # overall/district coverage pages.
        satellite_img = find_matching_file(
            SATELLITE_DIR,
            district,
            SATELLITE_SUFFIXES
        )

        if satellite_img is None:
            print(
                "WARNING: Pre-flood satellite image not found for:",
                district
            )

        lulc_img = (
            find_matching_file(
                LULC_MAP_DIR,
                district,
                LULC_SUFFIXES
            )
        )

        district_products.append({
            "district":
                district,

            "rc_df":
                rc_df,

            "village_df":
                village_df,

            "satellite_img":
                satellite_img,

            "flood_img":
                flood_map,

            "lulc_img":
                lulc_img,

            "coverage_img":
                district_coverage_lookup
                .get(
                    district_key
                ),
        })

        if not rc_df.empty:

            temp_rc = rc_df.copy()

            temp_rc[
                "District_Name"
            ] = district

            all_rc.append(
                temp_rc
            )

        if not village_df.empty:

            temp_village = (
                village_df.copy()
            )

            temp_village[
                "District"
            ] = district

            all_village.append(
                temp_village
            )

        all_summary.append(
            summary
        )

    district_summary_df = (
        pd.DataFrame(
            all_summary
        )
    )

    (
        acquisition_date,
        satellite_data,
        acquisition_time
    ) = read_acquisition_information(
        README_PATTERN
    )

    # ========================================================
    # PAGE 1 - EXECUTIVE SUMMARY
    # ========================================================

    add_executive_summary(
        doc,
        district_summary_df,
        acquisition_date,
        satellite_data,
        acquisition_time
    )

    # ========================================================
    # PAGE 2 - OVERALL ASSAM SAR COVERAGE
    # This occupies the FIRST blank page from the old report.
    # ========================================================

    doc.add_page_break()

    overall_caption = (
        f"Figure 1: Coverage of "
        f"{satellite_data} imageries "
        f"on {acquisition_date}"
    )

    add_report_image_page(
        doc,
        overall_coverage_img,
        caption=overall_caption,
        max_width_inches=9.0,
        max_height_inches=6.15
    )

    # ========================================================
    # PAGE 3 onward - INTERLEAVED DISTRICT SECTIONS
    #
    # Required report order:
    #
    #   District 1 coverage page
    #   District 1 detailed report
    #
    #   District 2 coverage page
    #   District 2 detailed report
    #
    #   District 3 coverage page
    #   District 3 detailed report
    #   ...
    #
    # Therefore district coverage pages are NOT grouped together.
    # Each coverage page appears immediately before the report
    # for the same district.
    # ========================================================

    for idx, item in enumerate(
        district_products
    ):

        # ----------------------------------------------------
        # A. DISTRICT-WISE SATELLITE COVERAGE PAGE
        # ----------------------------------------------------

        doc.add_page_break()

       # coverage_caption = (
         #   f"Area Coverage of "
         #   f"{item['district'].title()} District "
         #   f"from {satellite_data} imageries "
          #  f"on {acquisition_date}"
       # )

        add_report_image_page(
            doc,
            item["coverage_img"],
         #   caption=coverage_caption,
            max_width_inches=8.8,
            max_height_inches=6.05
        )

        # ----------------------------------------------------
        # B. SAME DISTRICT'S DETAILED REPORT
        #
        # Start it immediately after its coverage page.
        #
        # Original report layout:
        #   Pre-flood satellite image | Inundation map
        #   LULC map                  | Inundation map
        #
        # followed by RC/LULC and village statistics.
        # ----------------------------------------------------

        doc.add_page_break()

        add_district_to_report(
            doc,
            item["district"],
            item["rc_df"],
            item["village_df"],
            item["satellite_img"],
            item["flood_img"],
            item["lulc_img"]
        )

        # Do NOT insert an extra blank page here.
        # The next loop iteration begins with doc.add_page_break(),
        # which starts the next district's coverage page.

    add_end_marker(
        doc
    )

    # ========================================================
    # SAVE OUTPUTS
    # ========================================================

    doc.save(
        OUTPUT_DOCX
    )

    # Convert the FINAL document - including title pages - to PDF.
    convert_docx_to_pdf(
        OUTPUT_DOCX,
        OUTPUT_PDF
    )

    if all_rc:

        pd.concat(
            all_rc,
            ignore_index=True
        ).to_excel(
            OUTPUT_RC_EXCEL,
            index=False
        )

    if all_village:

        pd.concat(
            all_village,
            ignore_index=True
        ).to_excel(
            OUTPUT_VILLAGE_EXCEL,
            index=False
        )

    district_summary_df.to_excel(
        OUTPUT_DISTRICT_EXCEL,
        index=False
    )

    print(
        "\n"
        + "=" * 72
    )

    print(
        "REPORT GENERATION COMPLETE"
    )

    print(
        "=" * 72
    )

    print(
        "Word report       :",
        OUTPUT_DOCX
    )

    print(
        "PDF report        :",
        OUTPUT_PDF
    )

    print(
        "Overall coverage  :",
        overall_coverage_img
    )

    print(
        "RC statistics     :",
        OUTPUT_RC_EXCEL
    )

    print(
        "Village statistics:",
        OUTPUT_VILLAGE_EXCEL
    )

    print(
        "District summary  :",
        OUTPUT_DISTRICT_EXCEL
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    generate_report()



DATE AUTOMATICALLY READ FROM README
README date     : 01-Sep-2026
Processing date : 01-09-2026

Flood ZIP selected from README date:
Z:\Flood Inundation\DATA\as_2026_01_09_1800.zip

Flood ZIP extracted to:
Z:\Flood Inundation\DATA\extracted\20260901

Flood shapefile selected:
Z:\Flood Inundation\DATA\extracted\20260901\as_2026_01_09_1800.shp

Flood ZIP extracted to:
Z:\Flood Inundation\DATA\extracted

DATE AUTOMATICALLY READ FROM README
README date     : 01-Sep-2026
Processing date : 01-09-2026
READING GIS LAYERS

ORIGINAL CRS
District : EPSG:32646
RC       : EPSG:32646
Village  : EPSG:32646
Flood    : EPSG:4326
LULC     : EPSG:32646

ORIGINAL BOUNDS
District : [ 170028.02355941 2669402.51816381  798546.73744456 3097246.75000105]
Flood    : [92.65139104 25.71098061 94.72189275 27.29011082]

BOUNDS AFTER CRS ALIGNMENT
District : [ 170028.02355941 2669402.51816381  798546.73744456 3097246.75000105]
Flood    : [ 465248.64633827 2843685.00714492  670845.59465472 3019309.23303648]

Boundin

C:\Users\Standard\AppData\Local\Temp\ipykernel_32372\2858309433.py:3052: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  flood_union = flood_gdf.geometry.unary_union



Applying minimum district inundation-area filter:
  Threshold : 100.00 ha
  KEEP   BISWANATH                    1096.56 ha
  REJECT DIMA HASAO                      7.98 ha
  KEEP   GOLAGHAT                      417.70 ha
  KEEP   HOJAI                         571.83 ha
  KEEP   JORHAT                        593.61 ha
  REJECT KARBI ANGLONG                   1.97 ha
  KEEP   LAKHIMPUR                     116.76 ha
  KEEP   NAGAON                       1175.30 ha
  KEEP   SIBSAGAR                     2021.49 ha
  REJECT WEST KARBI ANGLONG              2.70 ha

Affected districts:
  - BISWANATH
  - GOLAGHAT
  - HOJAI
  - JORHAT
  - LAKHIMPUR
  - NAGAON
  - SIBSAGAR

AUTOMATIC SATELLITE FOOTPRINT SEARCH
README satellite : SENTINEL -1D SAR
Platform         : S1D
Source           : CDSE
Collection       : sentinel-1-grd
Search period    : 2026-09-01T00:00:00Z/2026-09-01T23:59:59Z
Matching scenes found: 5

CDSE Sentinel-1 COG access requires S3 credentials.
Raster URL: s3://eodata/Sentinel-1

C:\Users\Standard\AppData\Local\Temp\ipykernel_32372\2858309433.py:2776: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union


    RC: GOHPUR CIRCLE


...


[7/7] Processing SIBSAGAR
    RC: AMGURI CIRCLE


C:\Users\Standard\AppData\Local\Temp\ipykernel_32372\2858309433.py:2776: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union


    RC: SIBSAGAR CIRCLE


C:\Users\Standard\AppData\Local\Temp\ipykernel_32372\2858309433.py:2776: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union



README selected:
Z:\Flood Inundation\DATA\README.txt

Acquisition information:
Date      : 01-Sep-2026
Satellite : SENTINEL -1D SAR
Time      : 1800 Hrs

CONVERTING FINAL REPORT TO PDF
DOCX: Z:\Flood Inundation\01092026\01-09-2026\RC_village_Report_01-09-2026.docx
PDF : Z:\Flood Inundation\01092026\01-09-2026\RC_village_Report_01-09-2026.pdf
PDF conversion successful using Microsoft Word.

REPORT GENERATION COMPLETE
Word report       : Z:\Flood Inundation\01092026\01-09-2026\RC_village_Report_01-09-2026.docx
PDF report        : Z:\Flood Inundation\01092026\01-09-2026\RC_village_Report_01-09-2026.pdf
Overall coverage  : Z:\Flood Inundation\01092026\01-09-2026\sar_coverage_products\S1D_SAR_20260901_Overall_Assam_Coverage.png
RC statistics     : Z:\Flood Inundation\01092026\01-09-2026\RC_statistics_01-09-2026.xlsx
Village statistics: Z:\Flood Inundation\01092026\01-09-2026\Village_statistics_01-09-2026.xlsx
District summary  : Z:\Flood Inundation\01092026\01-09-2026\District_summary_01-0

In [9]:
CDSE_S3_ACCESS_KEY = "CXDOV1K1EWBA9ZN9WQC4"
CDSE_S3_SECRET_KEY = "o8wuwz4gCtMeil2GggY6CZF2xMDdgqOLhmB1YRd7"

In [10]:
pip install rasterio boto3

Note: you may need to restart the kernel to use updated packages.
